# Target Lock HUD v2

In [5]:
# Target Lock HUD v2 — animated HUD overlay, transparent WebM



# Output: media-site/animations/HUD/Target_Lock/target_lock_hud_v2_alpha.webm







from pathlib import Path







import numpy as np



import matplotlib.pyplot as plt



from matplotlib.patches import Circle, Arc, Rectangle



from matplotlib.lines import Line2D







from vizlib.animation_export import export_animation











OUTPUT_FORMAT = "webm"   # webm | mp4 | gif



ALPHA = False







OUT_DIR = Path("media-site/animations/HUD/Target_Lock")







W, H = 16, 9



DPI = 120



FPS = 24



DURATION = 6



FRAMES = FPS * DURATION







COL = "#35f6ff"



COL_DIM = "#1b7f88"



COL_WARN = "#ff4d4d"







LOOPS_ROTATION = 2



LOOPS_PULSE = 4



LOOPS_DOTS = 3



LOOPS_BARS = 5



LOOPS_BLINK = 6











fig, ax = plt.subplots(figsize=(W, H), dpi=DPI)







if ALPHA:



    fig.patch.set_facecolor((0, 0, 0, 0))



    fig.patch.set_alpha(0.0)



    ax.set_facecolor((0, 0, 0, 0))



else:



    fig.patch.set_facecolor("black")



    ax.set_facecolor("black")







ax.set_xlim(-16, 16)



ax.set_ylim(-9, 9)



ax.set_aspect("equal")



ax.axis("off")











# Main rings



rings = [



    Circle((0, 0), 3.2, fill=False, lw=1.4, ec=COL, alpha=0.75),



    Circle((0, 0), 4.35, fill=False, lw=0.8, ec=COL_DIM, alpha=0.55),



    Circle((0, 0), 5.55, fill=False, lw=0.7, ec=COL_DIM, alpha=0.35),



]



for ring in rings:



    ax.add_patch(ring)











# Crosshair



cross_lines = []



for x1, y1, x2, y2 in [



    (-6.7, 0, -3.7, 0),



    (3.7, 0, 6.7, 0),



    (0, -6.7, 0, -3.7),



    (0, 3.7, 0, 6.7),



]:



    line = Line2D([x1, x2], [y1, y2], lw=1.1, color=COL, alpha=0.75)



    ax.add_line(line)



    cross_lines.append(line)











# Rotating arcs



arcs = []



for radius, theta1, theta2, lw, alpha in [



    (6.2, 15, 82, 1.4, 0.85),



    (6.2, 195, 262, 1.4, 0.85),



    (4.85, 105, 160, 0.9, 0.55),



    (4.85, 285, 340, 0.9, 0.55),



]:



    arc = Arc(



        (0, 0),



        radius * 2,



        radius * 2,



        angle=0,



        theta1=theta1,



        theta2=theta2,



        lw=lw,



        color=COL,



        alpha=alpha,



    )



    ax.add_patch(arc)



    arcs.append(arc)











# Corner brackets



brackets = []



s = 2.25



gap = 0.55







for sx, sy in [(-1, -1), (-1, 1), (1, -1), (1, 1)]:



    x = sx * s



    y = sy * s







    l1 = Line2D([x, x + sx * -gap], [y, y], lw=1.3, color=COL)



    l2 = Line2D([x, x], [y, y + sy * -gap], lw=1.3, color=COL)







    ax.add_line(l1)



    ax.add_line(l2)







    brackets.extend([l1, l2])











# Sweep line



sweep, = ax.plot([], [], lw=1.2, color=COL, alpha=0.85)











# Moving scan dots



dots = []



for _ in range(18):



    dot, = ax.plot([], [], "o", ms=2.4, color=COL, alpha=0.0)



    dots.append(dot)











# Telemetry text



txt_main = ax.text(



    -14.7,



    7.4,



    "TARGET LOCK // ACTIVE",



    color=COL,



    fontsize=13,



    family="monospace",



    alpha=0.9,



)







txt_coord = ax.text(



    -14.7,



    6.75,



    "",



    color=COL_DIM,



    fontsize=10,



    family="monospace",



    alpha=0.85,



)







txt_range = ax.text(



    8.0,



    -7.5,



    "",



    color=COL,



    fontsize=10,



    family="monospace",



    alpha=0.85,



)







txt_warn = ax.text(



    -1.75,



    -7.25,



    "LOCK",



    color=COL_WARN,



    fontsize=14,



    family="monospace",



    alpha=0.0,



)











# Bottom data bars



bars = []



for i in range(18):



    rect = Rectangle(



        (-14.7 + i * 0.42, -7.55),



        0.28,



        0.18,



        fill=True,



        color=COL,



        alpha=0.2,



    )



    ax.add_patch(rect)



    bars.append(rect)











def polar_point(radius: float, angle_deg: float) -> tuple[float, float]:



    angle_rad = np.deg2rad(angle_deg)



    return radius * np.cos(angle_rad), radius * np.sin(angle_rad)











def update(frame: int) -> None:



    t = frame / FRAMES



    tau = 2 * np.pi







    pulse = 0.5 + 0.5 * np.sin(tau * LOOPS_PULSE * t)



    angle = 360 * LOOPS_ROTATION * t







    # Ring pulse



    for i, ring in enumerate(rings):



        ring.set_alpha(0.25 + 0.35 * pulse + i * 0.08)



        ring.set_linewidth(0.8 + 0.5 * pulse)







    # Rotating arcs — seamless



    for i, arc in enumerate(arcs):



        arc.angle = 360 * (LOOPS_ROTATION + i) * t



        arc.set_alpha(0.35 + 0.45 * pulse)







    # Brackets breathe



    for bracket in brackets:



        bracket.set_alpha(0.55 + 0.45 * pulse)







    # Sweep



    x0, y0 = polar_point(0.5, angle)



    x1, y1 = polar_point(5.75, angle)







    sweep.set_data([x0, x1], [y0, y1])



    sweep.set_alpha(0.3 + 0.6 * pulse)







    # Dots orbit/noise — seamless



    for i, dot in enumerate(dots):



        dot_phase = tau * LOOPS_DOTS * t + i







        dot_angle = angle + i * 23 + 12 * np.sin(dot_phase)



        radius = 2.4 + (i % 5) * 0.62 + 0.18 * np.sin(dot_phase)







        x, y = polar_point(radius, dot_angle)







        dot.set_data([x], [y])



        dot.set_alpha(0.15 + 0.55 * (0.5 + 0.5 * np.sin(dot_phase)))







    # Text telemetry — seamless



    jitter_ra = 184.732 + 0.015 * np.sin(tau * LOOPS_DOTS * t)



    jitter_dec = -12.448 + 0.012 * np.cos(tau * LOOPS_DOTS * t)







    txt_coord.set_text(



        f"RA {jitter_ra:08.3f}  "



        f"DEC {jitter_dec:+07.3f}  "



        f"SIG {78 + int(20 * pulse):02d}%"



    )







    txt_range.set_text(



        f"RANGE {42.7 + 0.8 * np.sin(tau * LOOPS_DOTS * t):05.1f} AU  VECTOR STABLE"



    )







    # Lock blink — seamless smooth blink



    blink = 0.5 + 0.5 * np.sin(tau * LOOPS_BLINK * t)



    txt_warn.set_alpha(0.25 + 0.6 * blink)







    # Bars — seamless



    for i, bar in enumerate(bars):



        bar_phase = tau * LOOPS_BARS * t + i * 0.8



        bar.set_alpha(0.15 + 0.65 * ((np.sin(bar_phase) + 1) / 2))











frames = []







for frame_idx in range(FRAMES):



    update(frame_idx)



    fig.canvas.draw()







    frame_rgba = np.asarray(fig.canvas.renderer.buffer_rgba())

    if ALPHA:

        frames.append(frame_rgba.copy())

    else:

        frames.append(frame_rgba[:, :, :3].copy())







plt.close(fig)











out_file = export_animation(



    frames=frames,



    out_dir=OUT_DIR,



    animation_name="target_lock_hud_v2",



    output_format=OUTPUT_FORMAT,



    fps=FPS,



    alpha=ALPHA,



)







print(f"Saved: {out_file}")



print(f"Frames: {len(frames)}")



print(f"Size: {out_file.stat().st_size / 1024:.1f} KB")



ffmpeg version 7.1.1 Copyright (c) 2000-2025 the FFmpeg developers
  built with clang version 18.1.8
  configuration: --prefix=/Users/mloktionov/anaconda3/envs/astro-ai --cc=arm64-apple-darwin20.0.0-clang --cxx=arm64-apple-darwin20.0.0-clang++ --nm=arm64-apple-darwin20.0.0-nm --ar=arm64-apple-darwin20.0.0-ar --disable-doc --enable-openssl --enable-demuxer=dash --enable-hardcoded-tables --enable-libfreetype --enable-libharfbuzz --enable-libfontconfig --enable-libopenh264 --enable-libdav1d --enable-cross-compile --arch=arm64 --target-os=darwin --cross-prefix=arm64-apple-darwin20.0.0- --host-cc=/Users/runner/miniforge3/conda-bld/ffmpeg_1748704173249/_build_env/bin/x86_64-apple-darwin13.4.0-clang --enable-neon --disable-gnutls --enable-libvpx --enable-libass --enable-pthreads --enable-libopenvino --enable-gpl --enable-libx264 --enable-libx265 --enable-libmp3lame --enable-libaom --enable-libsvtav1 --enable-libxml2 --enable-pic --enable-shared --disable-static --enable-version3 --enable-zlib

Saved: animations/HUD/Target_Lock/target_lock_hud_v2.webm
Frames: 144
Size: 993.6 KB


[out#0/webm @ 0x152705aa0] video:481KiB audio:0KiB subtitle:0KiB other streams:0KiB global headers:0KiB muxing overhead: 106.771040%
frame=  144 fps= 29 q=32.0 Lsize=     994KiB time=00:00:06.00 bitrate=1356.6kbits/s speed= 1.2x    


# Data Scan Panel v1

In [6]:
# Data Scan Panel v1 — animated transparent HUD side panel















# Output: media-site/animations/HUD/Data_Scan_Panel/data_scan_panel_v1.webm































from pathlib import Path































import numpy as np















import matplotlib.pyplot as plt















from matplotlib.patches import Rectangle















from matplotlib.lines import Line2D































from vizlib.animation_export import export_animation















































OUTPUT_FORMAT = "webm"   # webm | mp4 | gif















ALPHA = False































OUT_DIR = Path("media-site/animations/HUD/Data_Scan_Panel")































W, H = 16, 9















DPI = 120















FPS = 24















DURATION = 6















FRAMES = FPS * DURATION































COL = "#35f6ff"















COL_DIM = "#1b7f88"















COL_WARN = "#ff4d4d"















COL_SOFT = "#9ffcff"































LOOPS_SWEEP = 2















LOOPS_PULSE = 4















LOOPS_LINES = 3















LOOPS_BARS = 5















LOOPS_GLITCH = 6















































fig, ax = plt.subplots(figsize=(W, H), dpi=DPI)































if ALPHA:















    fig.patch.set_facecolor((0, 0, 0, 0))















    fig.patch.set_alpha(0.0)















    ax.set_facecolor((0, 0, 0, 0))















else:















    fig.patch.set_facecolor("black")















    ax.set_facecolor("black")































ax.set_xlim(0, 16)















ax.set_ylim(0, 9)















ax.set_aspect("equal")















ax.axis("off")















































# Panel base geometry















panel_x = 0.65















panel_y = 0.75















panel_w = 5.2















panel_h = 7.5































panel_bg = Rectangle(















    (panel_x, panel_y),















    panel_w,















    panel_h,















    fill=False,















    facecolor=COL,















    alpha=0.055,















    edgecolor=COL,















    linewidth=1.2,















)















ax.add_patch(panel_bg)































panel_inner = Rectangle(















    (panel_x + 0.18, panel_y + 0.18),















    panel_w - 0.36,















    panel_h - 0.36,















    fill=False,















    edgecolor=COL_DIM,















    linewidth=0.8,















    alpha=0.55,















)















ax.add_patch(panel_inner)















































# Corner accents















corner_lines = []















corner_len = 0.55































for sx, sy in [















    (panel_x, panel_y),















    (panel_x + panel_w, panel_y),















    (panel_x, panel_y + panel_h),















    (panel_x + panel_w, panel_y + panel_h),















]:















    left = sx == panel_x















    bottom = sy == panel_y































    x_dir = 1 if left else -1















    y_dir = 1 if bottom else -1































    h_line = Line2D(















        [sx, sx + x_dir * corner_len],















        [sy, sy],















        lw=1.8,















        color=COL,















        alpha=0.9,















    )















    v_line = Line2D(















        [sx, sx],















        [sy, sy + y_dir * corner_len],















        lw=1.8,















        color=COL,















        alpha=0.9,















    )































    ax.add_line(h_line)















    ax.add_line(v_line)















    corner_lines.extend([h_line, v_line])















































# Header















title = ax.text(















    panel_x + 0.35,















    panel_y + panel_h - 0.65,















    "DATA SCAN // OBJECT PROFILE",















    color=COL,















    fontsize=11,















    family="monospace",















    alpha=0.95,















)































subtitle = ax.text(















    panel_x + 0.35,















    panel_y + panel_h - 1.05,















    "SPECTRAL / POSITIONAL / SIGNAL LOCK",















    color=COL_DIM,















    fontsize=8,















    family="monospace",















    alpha=0.85,















)















































# Scanline















scanline = Line2D(















    [panel_x + 0.25, panel_x + panel_w - 0.25],















    [panel_y + panel_h - 1.35, panel_y + panel_h - 1.35],















    lw=1.2,















    color=COL,















    alpha=0.75,















)















ax.add_line(scanline)















































# Telemetry rows















rows = [















    ("CLASS", "COMPACT SOURCE"),















    ("RA", "184.732"),















    ("DEC", "-12.448"),















    ("DIST", "42.7 AU"),















    ("MAG", "+18.4"),















    ("VEL", "31.8 km/s"),















    ("TEMP", "142 K"),















    ("SIGNAL", "LOCKED"),















]































row_texts = []















row_y0 = panel_y + panel_h - 1.75































for i, (key, value) in enumerate(rows):















    y = row_y0 - i * 0.52































    key_txt = ax.text(















        panel_x + 0.38,















        y,















        f"{key:<7}",















        color=COL_DIM,















        fontsize=8.5,















        family="monospace",















        alpha=0.82,















    )































    value_txt = ax.text(















        panel_x + 1.85,















        y,















        value,















        color=COL_SOFT,















        fontsize=8.5,















        family="monospace",















        alpha=0.92,















    )































    row_texts.append((key_txt, value_txt))















































# Spectral strip















spec_x = panel_x + 0.4















spec_y = panel_y + 1.2















spec_w = panel_w - 0.8















spec_h = 1.05































spec_box = Rectangle(















    (spec_x, spec_y),















    spec_w,















    spec_h,















    fill=False,















    edgecolor=COL_DIM,















    linewidth=0.8,















    alpha=0.65,















)















ax.add_patch(spec_box)































spectrum_lines = []















n_lines = 64































for i in range(n_lines):















    x = spec_x + spec_w * i / (n_lines - 1)















    base = 0.12 + 0.75 * (















        0.5















        + 0.5 * np.sin(i * 0.37)















    )















    line = Line2D(















        [x, x],















        [spec_y + 0.12, spec_y + 0.12 + base],















        lw=0.8,















        color=COL,















        alpha=0.35,















    )















    ax.add_line(line)















    spectrum_lines.append(line)















































# Spectrum moving cursor















spec_cursor = Line2D(















    [spec_x, spec_x],















    [spec_y + 0.08, spec_y + spec_h - 0.08],















    lw=1.2,















    color=COL_WARN,















    alpha=0.9,















)















ax.add_line(spec_cursor)















































# Bottom signal bars















signal_bars = []















bar_x = panel_x + 0.38















bar_y = panel_y + 0.55































for i in range(18):















    rect = Rectangle(















        (bar_x + i * 0.235, bar_y),















        0.14,















        0.22 + 0.25 * ((i % 5) / 5),















        fill=True,















        color=COL,















        alpha=0.25,















    )















    ax.add_patch(rect)















    signal_bars.append(rect)















































# Small right-side marker ticks















ticks = []















tick_x = panel_x + panel_w - 0.42































for i in range(12):















    y = panel_y + 0.85 + i * 0.52















    tick = Line2D(















        [tick_x, tick_x + 0.22],















        [y, y],















        lw=0.8,















        color=COL_DIM,















        alpha=0.45,















    )















    ax.add_line(tick)















    ticks.append(tick)















































def update(frame: int) -> None:















    t = frame / FRAMES















    tau = 2 * np.pi































    pulse = 0.5 + 0.5 * np.sin(tau * LOOPS_PULSE * t)















    sweep_phase = 0.5 + 0.5 * np.sin(tau * LOOPS_SWEEP * t - np.pi / 2)































    panel_bg.set_alpha(0.04 + 0.035 * pulse)















    panel_inner.set_alpha(0.38 + 0.28 * pulse)































    for line in corner_lines:















        line.set_alpha(0.55 + 0.4 * pulse)































    title.set_alpha(0.78 + 0.2 * pulse)















    subtitle.set_alpha(0.45 + 0.35 * pulse)































    # Horizontal scanline moves down/up seamlessly















    scan_y = panel_y + 0.55 + (panel_h - 1.1) * sweep_phase















    scanline.set_ydata([scan_y, scan_y])















    scanline.set_alpha(0.25 + 0.55 * pulse)































    # Telemetry changing values















    ra = 184.732 + 0.015 * np.sin(tau * LOOPS_LINES * t)















    dec = -12.448 + 0.012 * np.cos(tau * LOOPS_LINES * t)















    dist = 42.7 + 0.8 * np.sin(tau * LOOPS_LINES * t)















    vel = 31.8 + 0.5 * np.cos(tau * LOOPS_LINES * t)















    signal = 78 + int(20 * pulse)































    dynamic_values = [















        "COMPACT SOURCE",















        f"{ra:08.3f}",















        f"{dec:+07.3f}",















        f"{dist:05.1f} AU",















        "+18.4",















        f"{vel:04.1f} km/s",















        "142 K",















        f"LOCKED {signal:02d}%",















    ]































    for i, (_, value_txt) in enumerate(row_texts):















        phase = tau * LOOPS_GLITCH * t + i * 0.7















        value_txt.set_text(dynamic_values[i])















        value_txt.set_alpha(0.62 + 0.3 * (0.5 + 0.5 * np.sin(phase)))































    # Spectrum animation















    cursor_x = spec_x + spec_w * ((LOOPS_SWEEP * t) % 1.0)















    spec_cursor.set_xdata([cursor_x, cursor_x])















    spec_cursor.set_alpha(0.45 + 0.45 * pulse)































    for i, line in enumerate(spectrum_lines):















        phase = tau * LOOPS_LINES * t + i * 0.29















        height = 0.12 + 0.78 * (0.5 + 0.5 * np.sin(phase))















        x = spec_x + spec_w * i / (n_lines - 1)































        line.set_data(















            [x, x],















            [spec_y + 0.12, spec_y + 0.12 + height],















        )















        line.set_alpha(0.18 + 0.45 * (0.5 + 0.5 * np.sin(phase + 1.1)))































    # Signal bars















    for i, bar in enumerate(signal_bars):















        phase = tau * LOOPS_BARS * t + i * 0.65















        bar.set_alpha(0.15 + 0.7 * (0.5 + 0.5 * np.sin(phase)))















        bar.set_height(0.18 + 0.45 * (0.5 + 0.5 * np.sin(phase + 0.8)))































    # Marker ticks















    for i, tick in enumerate(ticks):















        phase = tau * LOOPS_BARS * t + i * 0.45















        tick.set_alpha(0.2 + 0.55 * (0.5 + 0.5 * np.sin(phase)))















































frames = []































for frame_idx in range(FRAMES):















    update(frame_idx)















    fig.canvas.draw()































    frame_rgba = np.asarray(fig.canvas.renderer.buffer_rgba())



    if ALPHA:



        frames.append(frame_rgba.copy())



    else:



        frames.append(frame_rgba[:, :, :3].copy())































plt.close(fig)















































out_file = export_animation(















    frames=frames,















    out_dir=OUT_DIR,















    animation_name="data_scan_panel_v1",















    output_format=OUTPUT_FORMAT,















    fps=FPS,















    alpha=ALPHA,















)































print(f"Saved: {out_file}")















print(f"Frames: {len(frames)}")















print(f"Size: {out_file.stat().st_size / 1024:.1f} KB")









ffmpeg version 7.1.1 Copyright (c) 2000-2025 the FFmpeg developers
  built with clang version 18.1.8
  configuration: --prefix=/Users/mloktionov/anaconda3/envs/astro-ai --cc=arm64-apple-darwin20.0.0-clang --cxx=arm64-apple-darwin20.0.0-clang++ --nm=arm64-apple-darwin20.0.0-nm --ar=arm64-apple-darwin20.0.0-ar --disable-doc --enable-openssl --enable-demuxer=dash --enable-hardcoded-tables --enable-libfreetype --enable-libharfbuzz --enable-libfontconfig --enable-libopenh264 --enable-libdav1d --enable-cross-compile --arch=arm64 --target-os=darwin --cross-prefix=arm64-apple-darwin20.0.0- --host-cc=/Users/runner/miniforge3/conda-bld/ffmpeg_1748704173249/_build_env/bin/x86_64-apple-darwin13.4.0-clang --enable-neon --disable-gnutls --enable-libvpx --enable-libass --enable-pthreads --enable-libopenvino --enable-gpl --enable-libx264 --enable-libx265 --enable-libmp3lame --enable-libaom --enable-libsvtav1 --enable-libxml2 --enable-pic --enable-shared --disable-static --enable-version3 --enable-zlib

Saved: animations/HUD/Data_Scan_Panel/data_scan_panel_v1.webm
Frames: 144
Size: 426.0 KB


[out#0/webm @ 0x1551045a0] video:57KiB audio:0KiB subtitle:0KiB other streams:0KiB global headers:0KiB muxing overhead: 648.056246%
frame=  144 fps= 32 q=32.0 Lsize=     426KiB time=00:00:06.00 bitrate= 581.6kbits/s speed=1.32x    


In [7]:
# Orbital Sweep Overlay v1 — animated transparent navigation HUD



# Output: media-site/animations/HUD/Orbital_Sweep/orbital_sweep_overlay_v1.webm







from pathlib import Path







import numpy as np



import matplotlib.pyplot as plt



from matplotlib.patches import Arc, Circle



from matplotlib.lines import Line2D







from vizlib.animation_export import export_animation











OUTPUT_FORMAT = "webm"   # webm | mp4 | gif



ALPHA = False







OUT_DIR = Path("media-site/animations/HUD/Orbital_Sweep")







W, H = 16, 9



DPI = 120



FPS = 24



DURATION = 6



FRAMES = FPS * DURATION







COL = "#35f6ff"



COL_DIM = "#1b7f88"



COL_WARN = "#ff4d4d"



COL_SOFT = "#9ffcff"







LOOPS_ORBIT = 1



LOOPS_SWEEP = 2



LOOPS_PULSE = 4



LOOPS_MARKERS = 3



LOOPS_GRID = 2











fig, ax = plt.subplots(figsize=(W, H), dpi=DPI)







if ALPHA:



    fig.patch.set_facecolor((0, 0, 0, 0))



    fig.patch.set_alpha(0.0)



    ax.set_facecolor((0, 0, 0, 0))



else:



    fig.patch.set_facecolor("black")



    ax.set_facecolor("black")







ax.set_xlim(-16, 16)



ax.set_ylim(-9, 9)



ax.set_aspect("equal")



ax.axis("off")











# Central navigation origin



origin = Circle((0, 0), 0.08, fill=True, color=COL, alpha=0.9)



ax.add_patch(origin)







origin_ring = Circle((0, 0), 0.45, fill=False, ec=COL, lw=0.9, alpha=0.55)



ax.add_patch(origin_ring)











# Elliptical orbital paths



orbit_specs = [



    (12.0, 4.2, 0, 0.75),



    (9.0, 3.0, 0, 0.45),



    (6.2, 2.0, 0, 0.28),



]







orbits = []



for width, height, angle, alpha in orbit_specs:



    orbit = Arc(



        (0, 0),



        width,



        height,



        angle=angle,



        theta1=0,



        theta2=360,



        lw=1.0,



        color=COL_DIM,



        alpha=alpha,



    )



    ax.add_patch(orbit)



    orbits.append(orbit)











# Highlighted trajectory arc



trajectory_arc = Arc(



    (0, 0),



    12.0,



    4.2,



    angle=0,



    theta1=25,



    theta2=145,



    lw=2.0,



    color=COL,



    alpha=0.9,



)



ax.add_patch(trajectory_arc)











# Sweep radial line



sweep_line = Line2D([], [], lw=1.1, color=COL, alpha=0.85)



ax.add_line(sweep_line)











# Moving target marker



target_marker = Circle((0, 0), 0.16, fill=False, ec=COL_WARN, lw=1.5, alpha=0.95)



target_dot = Circle((0, 0), 0.045, fill=True, color=COL_WARN, alpha=0.95)



ax.add_patch(target_marker)



ax.add_patch(target_dot)











# Maneuver nodes



node_angles = [35, 115, 205, 310]



nodes = []



node_labels = []







for idx, deg in enumerate(node_angles):



    a = np.deg2rad(deg)



    x = 6.0 * np.cos(a)



    y = 2.1 * np.sin(a)







    node = Circle((x, y), 0.12, fill=True, color=COL, alpha=0.65)



    ax.add_patch(node)



    nodes.append(node)







    label = ax.text(



        x + 0.22,



        y + 0.14,



        f"DV-{idx + 1}",



        color=COL_DIM,



        fontsize=8,



        family="monospace",



        alpha=0.7,



    )



    node_labels.append(label)











# Predicted vector line



vector_line = Line2D([], [], lw=1.0, color=COL_WARN, alpha=0.65)



ax.add_line(vector_line)











# Grid ticks / range rings



range_rings = []



for r in [2.0, 3.6, 5.2, 6.8]:



    ring = Circle((0, 0), r, fill=False, ec=COL_DIM, lw=0.55, alpha=0.16)



    ax.add_patch(ring)



    range_rings.append(ring)











radial_ticks = []



for deg in range(0, 360, 15):



    a = np.deg2rad(deg)



    r0, r1 = 6.95, 7.25



    line = Line2D(



        [r0 * np.cos(a), r1 * np.cos(a)],



        [r0 * np.sin(a), r1 * np.sin(a)],



        lw=0.55,



        color=COL_DIM,



        alpha=0.35,



    )



    ax.add_line(line)



    radial_ticks.append(line)











# Telemetry labels



txt_title = ax.text(



    -14.7,



    7.55,



    "ORBITAL SWEEP // NAVIGATION SOLUTION",



    color=COL,



    fontsize=12,



    family="monospace",



    alpha=0.95,



)







txt_state = ax.text(



    -14.7,



    7.0,



    "",



    color=COL_DIM,



    fontsize=9,



    family="monospace",



    alpha=0.85,



)







txt_solution = ax.text(



    7.1,



    -7.55,



    "",



    color=COL_SOFT,



    fontsize=9,



    family="monospace",



    alpha=0.85,



)







txt_warning = ax.text(



    -1.65,



    -7.35,



    "TRANSFER WINDOW",



    color=COL_WARN,



    fontsize=11,



    family="monospace",



    alpha=0.75,



)











def ellipse_point(width: float, height: float, angle_deg: float) -> tuple[float, float]:



    a = np.deg2rad(angle_deg)



    return (width / 2) * np.cos(a), (height / 2) * np.sin(a)











def update(frame: int) -> None:



    t = frame / FRAMES



    tau = 2 * np.pi







    pulse = 0.5 + 0.5 * np.sin(tau * LOOPS_PULSE * t)



    orbit_angle = 360 * LOOPS_ORBIT * t



    sweep_angle = 360 * LOOPS_SWEEP * t







    # Orbit glow



    for i, orbit in enumerate(orbits):



        phase = tau * LOOPS_GRID * t + i * 0.8



        orbit.set_alpha(0.18 + 0.42 * (0.5 + 0.5 * np.sin(phase)))







    origin.set_alpha(0.55 + 0.4 * pulse)



    origin_ring.set_alpha(0.25 + 0.45 * pulse)



    origin_ring.set_radius(0.42 + 0.12 * pulse)







    # Highlighted trajectory slowly rotates without jump



    trajectory_arc.angle = 360 * t



    trajectory_arc.set_alpha(0.45 + 0.45 * pulse)







    # Sweep line



    a = np.deg2rad(sweep_angle)



    sweep_line.set_data([0, 7.3 * np.cos(a)], [0, 7.3 * np.sin(a)])



    sweep_line.set_alpha(0.25 + 0.5 * pulse)







    # Moving target along outer ellipse



    tx, ty = ellipse_point(12.0, 4.2, orbit_angle + 25)



    target_marker.center = (tx, ty)



    target_dot.center = (tx, ty)



    target_marker.set_alpha(0.55 + 0.4 * pulse)



    target_dot.set_alpha(0.7 + 0.25 * pulse)







    # Predicted vector from origin to target



    vector_line.set_data([0, tx], [0, ty])



    vector_line.set_alpha(0.22 + 0.38 * pulse)







    # Nodes blink seamlessly



    for i, node in enumerate(nodes):



        phase = tau * LOOPS_MARKERS * t + i * 0.7



        node.set_alpha(0.28 + 0.6 * (0.5 + 0.5 * np.sin(phase)))



        node.set_radius(0.09 + 0.06 * (0.5 + 0.5 * np.sin(phase)))







    for i, label in enumerate(node_labels):



        phase = tau * LOOPS_MARKERS * t + i * 0.7



        label.set_alpha(0.35 + 0.45 * (0.5 + 0.5 * np.sin(phase)))







    # Range rings and ticks



    for i, ring in enumerate(range_rings):



        phase = tau * LOOPS_GRID * t + i * 0.55



        ring.set_alpha(0.08 + 0.18 * (0.5 + 0.5 * np.sin(phase)))







    for i, tick in enumerate(radial_ticks):



        phase = tau * LOOPS_SWEEP * t + i * 0.17



        tick.set_alpha(0.12 + 0.36 * (0.5 + 0.5 * np.sin(phase)))







    # Telemetry



    dv = 3.42 + 0.18 * np.sin(tau * LOOPS_ORBIT * t)



    eta = 18.6 + 1.4 * np.cos(tau * LOOPS_ORBIT * t)



    phase_deg = (orbit_angle + 25) % 360







    txt_state.set_text(



        f"PHASE {phase_deg:06.2f} DEG  DV {dv:04.2f} km/s  ETA {eta:04.1f} h"



    )







    txt_solution.set_text(



        f"TARGET VECTOR [{tx:+05.2f}, {ty:+05.2f}]  SOLUTION STABLE"



    )







    txt_warning.set_alpha(0.3 + 0.55 * pulse)











frames = []







for frame_idx in range(FRAMES):



    update(frame_idx)



    fig.canvas.draw()







    frame_rgba = np.asarray(fig.canvas.renderer.buffer_rgba())

    if ALPHA:

        frames.append(frame_rgba.copy())

    else:

        frames.append(frame_rgba[:, :, :3].copy())







plt.close(fig)











out_file = export_animation(



    frames=frames,



    out_dir=OUT_DIR,



    animation_name="orbital_sweep_overlay_v1",



    output_format=OUTPUT_FORMAT,



    fps=FPS,



    alpha=ALPHA,



)







print(f"Saved: {out_file}")



print(f"Frames: {len(frames)}")



print(f"Size: {out_file.stat().st_size / 1024:.1f} KB")



ffmpeg version 7.1.1 Copyright (c) 2000-2025 the FFmpeg developers
  built with clang version 18.1.8
  configuration: --prefix=/Users/mloktionov/anaconda3/envs/astro-ai --cc=arm64-apple-darwin20.0.0-clang --cxx=arm64-apple-darwin20.0.0-clang++ --nm=arm64-apple-darwin20.0.0-nm --ar=arm64-apple-darwin20.0.0-ar --disable-doc --enable-openssl --enable-demuxer=dash --enable-hardcoded-tables --enable-libfreetype --enable-libharfbuzz --enable-libfontconfig --enable-libopenh264 --enable-libdav1d --enable-cross-compile --arch=arm64 --target-os=darwin --cross-prefix=arm64-apple-darwin20.0.0- --host-cc=/Users/runner/miniforge3/conda-bld/ffmpeg_1748704173249/_build_env/bin/x86_64-apple-darwin13.4.0-clang --enable-neon --disable-gnutls --enable-libvpx --enable-libass --enable-pthreads --enable-libopenvino --enable-gpl --enable-libx264 --enable-libx265 --enable-libmp3lame --enable-libaom --enable-libsvtav1 --enable-libxml2 --enable-pic --enable-shared --disable-static --enable-version3 --enable-zlib

Saved: animations/HUD/Orbital_Sweep/orbital_sweep_overlay_v1.webm
Frames: 144
Size: 934.1 KB


[out#0/webm @ 0x123f05050] video:475KiB audio:0KiB subtitle:0KiB other streams:0KiB global headers:0KiB muxing overhead: 96.574855%
frame=  144 fps= 28 q=32.0 Lsize=     934KiB time=00:00:06.00 bitrate=1275.3kbits/s speed=1.19x    


# Radar Sweep

In [10]:
# Radar Sweep v1 — animated transparent HUD radar overlay







# Output: media-site/animations/HUD/Radar_Sweep/radar_sweep_v1.webm















from pathlib import Path















import numpy as np







import matplotlib.pyplot as plt







from matplotlib.patches import Circle, Wedge







from matplotlib.lines import Line2D















from vizlib.animation_export import export_animation























OUTPUT_FORMAT = "webm"   # webm | mp4 | gif







ALPHA = False















OUT_DIR = Path("media-site/animations/HUD/Radar_Sweep")















W, H = 10, 10







DPI = 120







FPS = 24







DURATION = 6







FRAMES = FPS * DURATION















COL = "#35f6ff"







COL_DIM = "#1b7f88"







COL_WARN = "#ff4d4d"







COL_SOFT = "#9ffcff"















LOOPS_SWEEP = 2







LOOPS_PULSE = 4







LOOPS_BLIPS = 3







LOOPS_NOISE = 6















fig, ax = plt.subplots(figsize=(W, H), dpi=DPI)















if ALPHA:







    fig.patch.set_facecolor((0, 0, 0, 0))







    fig.patch.set_alpha(0.0)







    ax.set_facecolor((0, 0, 0, 0))







else:







    fig.patch.set_facecolor("black")







    ax.set_facecolor("black")















ax.set_xlim(-8, 8)







ax.set_ylim(-8, 8)







ax.set_aspect("equal")







ax.axis("off")























CENTER = (0, 0)







R_MAX = 5.8























# Range rings







rings = []







for r in [1.45, 2.9, 4.35, 5.8]:







    ring = Circle(CENTER, r, fill=False, ec=COL_DIM, lw=0.8, alpha=0.28)







    ax.add_patch(ring)







    rings.append(ring)























# Cross grid







grid_lines = []







for angle_deg in range(0, 180, 30):







    a = np.deg2rad(angle_deg)







    x = R_MAX * np.cos(a)







    y = R_MAX * np.sin(a)















    line = Line2D([-x, x], [-y, y], lw=0.55, color=COL_DIM, alpha=0.22)







    ax.add_line(line)







    grid_lines.append(line)























# Outer circle accents







outer = Circle(CENTER, R_MAX, fill=False, ec=COL, lw=1.2, alpha=0.75)







ax.add_patch(outer)















inner_core = Circle(CENTER, 0.12, fill=True, color=COL, alpha=0.8)







ax.add_patch(inner_core)























# Sweep sector and leading line







sweep_sector = Wedge(







    CENTER,







    R_MAX,







    theta1=0,







    theta2=38,







    facecolor=COL,







    edgecolor=None,







    alpha=0.0,







    fill=False,







)







ax.add_patch(sweep_sector)















sweep_line = Line2D([], [], lw=1.4, color=COL, alpha=0.85)







ax.add_line(sweep_line)























# Blips: fixed positions, animated visibility when sweep passes







blip_defs = [







    (1.2, 25, 0.09, "A"),







    (2.7, 92, 0.12, "B"),







    (4.8, 145, 0.10, "C"),







    (3.9, 238, 0.14, "D"),







    (5.2, 318, 0.10, "E"),







    (2.1, 276, 0.08, "F"),







]















blips = []







blip_labels = []















for radius, angle_deg, size, label in blip_defs:







    a = np.deg2rad(angle_deg)







    x = radius * np.cos(a)







    y = radius * np.sin(a)















    dot = Circle((x, y), size, fill=True, color=COL_WARN, alpha=0.15)







    halo = Circle((x, y), size * 3.0, fill=False, ec=COL_WARN, lw=0.8, alpha=0.0)















    ax.add_patch(halo)







    ax.add_patch(dot)















    txt = ax.text(







        x + 0.22,







        y + 0.12,







        label,







        color=COL_WARN,







        fontsize=8,







        family="monospace",







        alpha=0.0,







    )















    blips.append((dot, halo, radius, angle_deg, size))







    blip_labels.append(txt)























# Background noise contacts







rng = np.random.default_rng(42)







noise_defs = []







noise_points = []















for _ in range(70):







    radius = R_MAX * np.sqrt(rng.uniform(0.05, 1.0))







    angle_deg = rng.uniform(0, 360)







    phase = rng.uniform(0, 2 * np.pi)















    a = np.deg2rad(angle_deg)







    x = radius * np.cos(a)







    y = radius * np.sin(a)















    point, = ax.plot([x], [y], "o", ms=rng.uniform(1.0, 2.0), color=COL, alpha=0.0)















    noise_defs.append((radius, angle_deg, phase))







    noise_points.append(point)























# Telemetry







txt_title = ax.text(







    -7.2,







    7.15,







    "RADAR SWEEP // LOCAL CONTACT MAP",







    color=COL,







    fontsize=12,







    family="monospace",







    alpha=0.95,







)















txt_state = ax.text(







    -7.2,







    6.6,







    "",







    color=COL_DIM,







    fontsize=9,







    family="monospace",







    alpha=0.85,







)















txt_contact = ax.text(







    -1.8,







    -7.2,







    "",







    color=COL_SOFT,







    fontsize=9,







    family="monospace",







    alpha=0.85,







)























def angle_distance_deg(a: float, b: float) -> float:







    return abs((a - b + 180) % 360 - 180)























def update(frame: int) -> None:







    t = frame / FRAMES







    tau = 2 * np.pi















    pulse = 0.5 + 0.5 * np.sin(tau * LOOPS_PULSE * t)







    sweep_angle = (360 * LOOPS_SWEEP * t) % 360















    # Base glow







    outer.set_alpha(0.45 + 0.35 * pulse)







    inner_core.set_alpha(0.55 + 0.4 * pulse)















    for i, ring in enumerate(rings):







        phase = tau * LOOPS_PULSE * t + i * 0.75







        ring.set_alpha(0.12 + 0.28 * (0.5 + 0.5 * np.sin(phase)))















    for i, line in enumerate(grid_lines):







        phase = tau * LOOPS_PULSE * t + i * 0.4







        line.set_alpha(0.08 + 0.22 * (0.5 + 0.5 * np.sin(phase)))















    # Sweep sector







    sweep_width = 42







    sweep_sector.set_theta1(sweep_angle - sweep_width)







    sweep_sector.set_theta2(sweep_angle)







    sweep_sector.set_alpha(0.05 + 0.08 * pulse)















    a = np.deg2rad(sweep_angle)







    sweep_line.set_data([0, R_MAX * np.cos(a)], [0, R_MAX * np.sin(a)])







    sweep_line.set_alpha(0.35 + 0.55 * pulse)















    # Blips light up when sweep crosses them







    active_label = "NONE"















    for i, (dot, halo, radius, angle_deg, size) in enumerate(blips):







        dist = angle_distance_deg(sweep_angle, angle_deg)







        hit = np.exp(-(dist / 18.0) ** 2)















        base_phase = tau * LOOPS_BLIPS * t + i * 0.9







        idle = 0.08 + 0.1 * (0.5 + 0.5 * np.sin(base_phase))















        dot_alpha = min(1.0, idle + 0.9 * hit)







        halo_alpha = min(0.8, 0.75 * hit)















        dot.set_alpha(dot_alpha)







        dot.set_radius(size * (1.0 + 1.4 * hit))















        halo.set_alpha(halo_alpha)







        halo.set_radius(size * (2.2 + 7.0 * hit))















        blip_labels[i].set_alpha(0.85 * hit)















        if hit > 0.65:







            active_label = blip_labels[i].get_text()















    # Noise flicker







    for i, point in enumerate(noise_points):







        _, angle_deg, phase = noise_defs[i]







        dist = angle_distance_deg(sweep_angle, angle_deg)







        sweep_hit = np.exp(-(dist / 30.0) ** 2)















        flicker = 0.5 + 0.5 * np.sin(tau * LOOPS_NOISE * t + phase)







        point.set_alpha(0.03 + 0.28 * sweep_hit * flicker)















    txt_state.set_text(







        f"AZ {sweep_angle:06.2f} DEG  RANGE {R_MAX:03.1f} AU  CONTACTS {len(blips):02d}"







    )















    txt_contact.set_text(







        f"ACTIVE CONTACT: {active_label:<4}  SIGNAL SWEEP NOMINAL"







    )























frames = []















for frame_idx in range(FRAMES):







    update(frame_idx)







    fig.canvas.draw()















    frame_rgba = np.asarray(fig.canvas.renderer.buffer_rgba())

    if ALPHA:

        frames.append(frame_rgba.copy())

    else:

        frames.append(frame_rgba[:, :, :3].copy())















plt.close(fig)























out_file = export_animation(







    frames=frames,







    out_dir=OUT_DIR,







    animation_name="radar_sweep_v1",







    output_format=OUTPUT_FORMAT,







    fps=FPS,







    alpha=ALPHA,







)















print(f"Saved: {out_file}")







print(f"Frames: {len(frames)}")







print(f"Size: {out_file.stat().st_size / 1024:.1f} KB")





ffmpeg version 7.1.1 Copyright (c) 2000-2025 the FFmpeg developers
  built with clang version 18.1.8
  configuration: --prefix=/Users/mloktionov/anaconda3/envs/astro-ai --cc=arm64-apple-darwin20.0.0-clang --cxx=arm64-apple-darwin20.0.0-clang++ --nm=arm64-apple-darwin20.0.0-nm --ar=arm64-apple-darwin20.0.0-ar --disable-doc --enable-openssl --enable-demuxer=dash --enable-hardcoded-tables --enable-libfreetype --enable-libharfbuzz --enable-libfontconfig --enable-libopenh264 --enable-libdav1d --enable-cross-compile --arch=arm64 --target-os=darwin --cross-prefix=arm64-apple-darwin20.0.0- --host-cc=/Users/runner/miniforge3/conda-bld/ffmpeg_1748704173249/_build_env/bin/x86_64-apple-darwin13.4.0-clang --enable-neon --disable-gnutls --enable-libvpx --enable-libass --enable-pthreads --enable-libopenvino --enable-gpl --enable-libx264 --enable-libx265 --enable-libmp3lame --enable-libaom --enable-libsvtav1 --enable-libxml2 --enable-pic --enable-shared --disable-static --enable-version3 --enable-zlib

Saved: animations/HUD/Radar_Sweep/radar_sweep_v1.webm
Frames: 144
Size: 939.8 KB


[out#0/webm @ 0x14fe25480] video:467KiB audio:0KiB subtitle:0KiB other streams:0KiB global headers:0KiB muxing overhead: 101.437013%
frame=  144 fps= 30 q=32.0 Lsize=     940KiB time=00:00:06.00 bitrate=1283.1kbits/s speed=1.23x    


# Spectral Analyzer

In [11]:
# Spectral Analyzer v1 — animated transparent spectral HUD















# Output: media-site/animations/HUD/Spectral_Analyzer/spectral_analyzer_v1.webm































from pathlib import Path































import numpy as np















import matplotlib.pyplot as plt















from matplotlib.patches import Rectangle















from matplotlib.lines import Line2D































from vizlib.animation_export import export_animation















































OUTPUT_FORMAT = "webm"   # webm | mp4 | gif















ALPHA = False































OUT_DIR = Path("media-site/animations/HUD/Spectral_Analyzer")































W, H = 16, 9















DPI = 120















FPS = 24















DURATION = 6















FRAMES = FPS * DURATION































COL = "#35f6ff"















COL_DIM = "#1b7f88"















COL_WARN = "#ff4d4d"















COL_SOFT = "#9ffcff"































LOOPS_SWEEP = 2















LOOPS_SIGNAL = 3















LOOPS_GRID = 4















LOOPS_PULSE = 5















































fig, ax = plt.subplots(figsize=(W, H), dpi=DPI)































if ALPHA:















    fig.patch.set_facecolor((0, 0, 0, 0))















    fig.patch.set_alpha(0.0)















    ax.set_facecolor((0, 0, 0, 0))















else:















    fig.patch.set_facecolor("black")















    ax.set_facecolor("black")































ax.set_xlim(0, 16)















ax.set_ylim(0, 9)















ax.axis("off")















































# Main analyzer frame















frame = Rectangle(















    (0.8, 1.0),















    14.4,















    7.0,















    fill=False,















    facecolor=COL,















    alpha=0.035,















    edgecolor=COL,















    linewidth=1.2,















)















ax.add_patch(frame)































inner = Rectangle(















    (1.0, 1.2),















    14.0,















    6.6,















    fill=False,















    edgecolor=COL_DIM,















    linewidth=0.8,















    alpha=0.55,















)















ax.add_patch(inner)















































# Horizontal grid















grid_lines = []































for i in range(9):















    y = 1.45 + i * 0.7































    line = Line2D(















        [1.1, 14.9],















        [y, y],















        lw=0.45,















        color=COL_DIM,















        alpha=0.18,















    )































    ax.add_line(line)















    grid_lines.append(line)















































# Vertical spectral divisions















verticals = []































for i in range(24):















    x = 1.2 + i * 0.57































    line = Line2D(















        [x, x],















        [1.3, 7.7],















        lw=0.4,















        color=COL_DIM,















        alpha=0.12,















    )































    ax.add_line(line)















    verticals.append(line)















































# Spectrum curve















spectrum_x = np.linspace(1.2, 14.8, 420)































spectrum_line, = ax.plot(















    [],















    [],















    lw=1.8,















    color=COL,















    alpha=0.9,















)































glow_line, = ax.plot(















    [],















    [],















    lw=4.5,















    color=COL,















    alpha=0.08,















)















































# Spectral peaks















peak_markers = []































peak_positions = [2.8, 4.6, 7.2, 9.7, 12.1, 13.6]































for px in peak_positions:















    line = Line2D(















        [px, px],















        [1.4, 7.6],















        lw=1.0,















        color=COL_WARN,















        alpha=0.18,















    )















    ax.add_line(line)















    peak_markers.append(line)















































# Moving scan cursor















scan_cursor = Line2D(















    [1.2, 1.2],















    [1.15, 7.85],















    lw=1.4,















    color=COL_SOFT,















    alpha=0.9,















)















ax.add_line(scan_cursor)















































# Bottom histogram bars















bars = []































for i in range(48):















    x = 1.15 + i * 0.285































    rect = Rectangle(















        (x, 1.15),















        0.16,















        0.35,















        fill=True,















        color=COL,















        alpha=0.22,















    )































    ax.add_patch(rect)















    bars.append(rect)















































# Telemetry text















txt_title = ax.text(















    1.1,















    8.15,















    "SPECTRAL ANALYZER // SIGNAL DECOMPOSITION",















    color=COL,















    fontsize=12,















    family="monospace",















    alpha=0.95,















)































txt_state = ax.text(















    1.1,















    7.75,















    "",















    color=COL_DIM,















    fontsize=9,















    family="monospace",















    alpha=0.82,















)































txt_peak = ax.text(















    10.8,















    0.55,















    "",















    color=COL_SOFT,















    fontsize=9,















    family="monospace",















    alpha=0.85,















)















































# Small spectral labels















labels = ["Ha", "Hb", "OIII", "Na", "CaK", "IR"]































label_texts = []































for px, label in zip(peak_positions, labels):















    txt = ax.text(















        px - 0.12,















        7.92,















        label,















        color=COL_WARN,















        fontsize=7,















        family="monospace",















        alpha=0.55,















    )































    label_texts.append(txt)















































def gaussian(x, mu, sigma):















    return np.exp(-((x - mu) ** 2) / (2 * sigma**2))















































def update(frame_idx: int) -> None:















    t = frame_idx / FRAMES















    tau = 2 * np.pi































    pulse = 0.5 + 0.5 * np.sin(tau * LOOPS_PULSE * t)































    # Background breathing















    frame.set_alpha(0.02 + 0.03 * pulse)















    inner.set_alpha(0.3 + 0.25 * pulse)































    for i, line in enumerate(grid_lines):















        phase = tau * LOOPS_GRID * t + i * 0.35















        line.set_alpha(0.06 + 0.16 * (0.5 + 0.5 * np.sin(phase)))































    for i, line in enumerate(verticals):















        phase = tau * LOOPS_GRID * t + i * 0.15















        line.set_alpha(0.03 + 0.12 * (0.5 + 0.5 * np.sin(phase)))































    # Animated spectrum















    signal_phase = tau * LOOPS_SIGNAL * t































    y = (















        4.1















        + 0.9 * np.sin(0.65 * spectrum_x + signal_phase)















        + 0.35 * np.sin(2.2 * spectrum_x + signal_phase * 1.3)















    )































    # Spectral peaks















    for i, peak_x in enumerate(peak_positions):















        amp = 0.8 + 0.35 * np.sin(signal_phase + i)















        y += amp * gaussian(spectrum_x, peak_x, 0.08 + 0.015 * i)































    spectrum_line.set_data(spectrum_x, y)















    glow_line.set_data(spectrum_x, y)































    spectrum_line.set_alpha(0.65 + 0.25 * pulse)















    glow_line.set_alpha(0.05 + 0.08 * pulse)































    # Cursor sweep















    cursor_phase = (LOOPS_SWEEP * t) % 1.0















    cursor_x = 1.2 + 13.6 * cursor_phase































    scan_cursor.set_xdata([cursor_x, cursor_x])















    scan_cursor.set_alpha(0.35 + 0.55 * pulse)































    # Peak markers react to cursor















    active_peak = "NONE"































    for i, line in enumerate(peak_markers):















        px = peak_positions[i]































        dist = abs(cursor_x - px)















        hit = np.exp(-(dist / 0.32) ** 2)































        line.set_alpha(0.12 + 0.7 * hit)































        label_texts[i].set_alpha(0.25 + 0.7 * hit)































        if hit > 0.65:















            active_peak = labels[i]































    # Histogram















    for i, bar in enumerate(bars):















        phase = tau * LOOPS_SIGNAL * t + i * 0.21































        h = 0.12 + 0.75 * (0.5 + 0.5 * np.sin(phase))































        bar.set_height(h)















        bar.set_alpha(0.08 + 0.45 * (0.5 + 0.5 * np.sin(phase + 0.7)))































    # Telemetry















    wavelength = 380 + 420 * cursor_phase















    snr = 28 + 9 * np.sin(signal_phase)































    txt_state.set_text(















        f"WAVELENGTH {wavelength:06.1f} nm   SNR {snr:04.1f}   SCAN ACTIVE"















    )































    txt_peak.set_text(















        f"ACTIVE FEATURE: {active_peak:<4}   RESOLUTION LOCKED"















    )















































frames = []































for frame_idx in range(FRAMES):















    update(frame_idx)































    fig.canvas.draw()































    frame_rgba = np.asarray(fig.canvas.renderer.buffer_rgba())



    if ALPHA:



        frames.append(frame_rgba.copy())



    else:



        frames.append(frame_rgba[:, :, :3].copy())































plt.close(fig)















































out_file = export_animation(















    frames=frames,















    out_dir=OUT_DIR,















    animation_name="spectral_analyzer_v1",















    output_format=OUTPUT_FORMAT,















    fps=FPS,















    alpha=ALPHA,















)































print(f"Saved: {out_file}")















print(f"Frames: {len(frames)}")















print(f"Size: {out_file.stat().st_size / 1024:.1f} KB")









ffmpeg version 7.1.1 Copyright (c) 2000-2025 the FFmpeg developers
  built with clang version 18.1.8
  configuration: --prefix=/Users/mloktionov/anaconda3/envs/astro-ai --cc=arm64-apple-darwin20.0.0-clang --cxx=arm64-apple-darwin20.0.0-clang++ --nm=arm64-apple-darwin20.0.0-nm --ar=arm64-apple-darwin20.0.0-ar --disable-doc --enable-openssl --enable-demuxer=dash --enable-hardcoded-tables --enable-libfreetype --enable-libharfbuzz --enable-libfontconfig --enable-libopenh264 --enable-libdav1d --enable-cross-compile --arch=arm64 --target-os=darwin --cross-prefix=arm64-apple-darwin20.0.0- --host-cc=/Users/runner/miniforge3/conda-bld/ffmpeg_1748704173249/_build_env/bin/x86_64-apple-darwin13.4.0-clang --enable-neon --disable-gnutls --enable-libvpx --enable-libass --enable-pthreads --enable-libopenvino --enable-gpl --enable-libx264 --enable-libx265 --enable-libmp3lame --enable-libaom --enable-libsvtav1 --enable-libxml2 --enable-pic --enable-shared --disable-static --enable-version3 --enable-zlib

Saved: animations/HUD/Spectral_Analyzer/spectral_analyzer_v1.webm
Frames: 144
Size: 1271.5 KB


[out#0/webm @ 0x12df04e10] video:359KiB audio:0KiB subtitle:0KiB other streams:0KiB global headers:0KiB muxing overhead: 253.908127%
frame=  144 fps= 19 q=32.0 Lsize=    1272KiB time=00:00:06.00 bitrate=1736.0kbits/s speed=0.794x    


# Cinematic Warning System v1

In [12]:
# Cinematic Warning System v1 — animated transparent alert HUD



# Output: media-site/animations/HUD/Warning_System/warning_system_v1.webm







from pathlib import Path







import numpy as np



import matplotlib.pyplot as plt



from matplotlib.patches import Rectangle, Circle



from matplotlib.lines import Line2D







from vizlib.animation_export import export_animation











OUTPUT_FORMAT = "webm"   # webm | mp4 | gif



ALPHA = False







OUT_DIR = Path("media-site/animations/HUD/Warning_System")







W, H = 16, 9



DPI = 120



FPS = 24



DURATION = 6



FRAMES = FPS * DURATION







COL = "#ff3b3b"



COL_DIM = "#7f1b1b"



COL_SOFT = "#ff9a9a"



COL_AMBER = "#ffb000"







LOOPS_PULSE = 4



LOOPS_SCAN = 2



LOOPS_GLITCH = 6



LOOPS_BARS = 5











fig, ax = plt.subplots(figsize=(W, H), dpi=DPI)







if ALPHA:



    fig.patch.set_facecolor((0, 0, 0, 0))



    fig.patch.set_alpha(0.0)



    ax.set_facecolor((0, 0, 0, 0))



else:



    fig.patch.set_facecolor("black")



    ax.set_facecolor("black")







ax.set_xlim(0, 16)



ax.set_ylim(0, 9)



ax.axis("off")











# Main alert frame



outer = Rectangle(



    (0.7, 0.7),



    14.6,



    7.6,



    fill=False,



    edgecolor=COL,



    linewidth=1.8,



    alpha=0.75,



)



ax.add_patch(outer)







inner = Rectangle(



    (1.05, 1.05),



    13.9,



    6.9,



    fill=False,



    edgecolor=COL_DIM,



    linewidth=0.9,



    alpha=0.45,



)



ax.add_patch(inner)











# Warning side bars



left_bar = Rectangle((0.35, 1.0), 0.16, 7.0, fill=True, color=COL, alpha=0.25)



right_bar = Rectangle((15.49, 1.0), 0.16, 7.0, fill=True, color=COL, alpha=0.25)



ax.add_patch(left_bar)



ax.add_patch(right_bar)











# Central warning title



txt_alert = ax.text(



    8.0,



    5.1,



    "COLLISION ALERT",



    color=COL,



    fontsize=34,



    family="monospace",



    ha="center",



    va="center",



    alpha=0.95,



)







txt_sub = ax.text(



    8.0,



    4.45,



    "TRAJECTORY CONVERGENCE // IMMEDIATE VECTOR CORRECTION REQUIRED",



    color=COL_SOFT,



    fontsize=10,



    family="monospace",



    ha="center",



    va="center",



    alpha=0.85,



)







txt_state = ax.text(



    1.35,



    7.45,



    "",



    color=COL_AMBER,



    fontsize=10,



    family="monospace",



    alpha=0.85,



)







txt_timer = ax.text(



    12.15,



    7.45,



    "",



    color=COL_SOFT,



    fontsize=10,



    family="monospace",



    alpha=0.85,



)











# Scan line



scan_line = Line2D([1.1, 14.9], [1.2, 1.2], lw=1.2, color=COL, alpha=0.7)



ax.add_line(scan_line)











# Corner blocks



corner_blocks = []



for x, y in [(0.7, 0.7), (14.8, 0.7), (0.7, 7.95), (14.8, 7.95)]:



    block = Rectangle((x, y), 0.5, 0.18, fill=True, color=COL, alpha=0.65)



    ax.add_patch(block)



    corner_blocks.append(block)











# Alarm indicators



lights = []



for i in range(7):



    x = 5.6 + i * 0.8



    light = Circle((x, 3.3), 0.12, fill=True, color=COL, alpha=0.25)



    ax.add_patch(light)



    lights.append(light)











# Bottom diagnostic bars



bars = []



for i in range(42):



    x = 1.35 + i * 0.32



    rect = Rectangle((x, 1.45), 0.18, 0.3, fill=True, color=COL, alpha=0.18)



    ax.add_patch(rect)



    bars.append(rect)











# Glitch fragments



glitch_lines = []



rng = np.random.default_rng(7)







for _ in range(20):



    x = rng.uniform(1.3, 14.5)



    y = rng.uniform(2.0, 7.0)



    length = rng.uniform(0.2, 1.2)







    line = Line2D([x, x + length], [y, y], lw=rng.uniform(0.6, 1.5), color=COL, alpha=0.0)



    ax.add_line(line)



    glitch_lines.append((line, rng.uniform(0, 2 * np.pi)))











def update(frame_idx: int) -> None:



    t = frame_idx / FRAMES



    tau = 2 * np.pi







    pulse = 0.5 + 0.5 * np.sin(tau * LOOPS_PULSE * t)



    hard_pulse = pulse ** 2.5



    scan_phase = 0.5 + 0.5 * np.sin(tau * LOOPS_SCAN * t - np.pi / 2)







    outer.set_alpha(0.35 + 0.55 * hard_pulse)



    inner.set_alpha(0.18 + 0.35 * pulse)







    left_bar.set_alpha(0.15 + 0.45 * hard_pulse)



    right_bar.set_alpha(0.15 + 0.45 * hard_pulse)







    txt_alert.set_alpha(0.45 + 0.5 * hard_pulse)



    txt_sub.set_alpha(0.35 + 0.45 * pulse)







    scan_y = 1.2 + 6.6 * scan_phase



    scan_line.set_ydata([scan_y, scan_y])



    scan_line.set_alpha(0.2 + 0.55 * pulse)







    for i, block in enumerate(corner_blocks):



        phase = tau * LOOPS_PULSE * t + i * 0.7



        block.set_alpha(0.25 + 0.55 * (0.5 + 0.5 * np.sin(phase)))







    for i, light in enumerate(lights):



        phase = tau * LOOPS_GLITCH * t + i * 0.9



        light.set_alpha(0.15 + 0.75 * (0.5 + 0.5 * np.sin(phase)))



        light.set_radius(0.09 + 0.08 * (0.5 + 0.5 * np.sin(phase)))







    for i, bar in enumerate(bars):



        phase = tau * LOOPS_BARS * t + i * 0.35



        h = 0.1 + 0.75 * (0.5 + 0.5 * np.sin(phase))



        bar.set_height(h)



        bar.set_alpha(0.08 + 0.42 * (0.5 + 0.5 * np.sin(phase + 0.7)))







    for line, phase0 in glitch_lines:



        phase = 0.5 + 0.5 * np.sin(tau * LOOPS_GLITCH * t + phase0)



        alpha = 0.55 if phase > 0.92 else 0.0



        line.set_alpha(alpha)







    range_km = 12840 - 950 * (0.5 + 0.5 * np.sin(tau * LOOPS_SCAN * t))



    eta = 41.2 - 3.8 * (0.5 + 0.5 * np.sin(tau * LOOPS_SCAN * t))







    txt_state.set_text(f"RANGE {range_km:07.1f} km  CLOSING RATE CRITICAL")



    txt_timer.set_text(f"ETA {eta:05.1f} s  STATUS RED")











frames = []







for frame_idx in range(FRAMES):



    update(frame_idx)



    fig.canvas.draw()







    frame_rgba = np.asarray(fig.canvas.renderer.buffer_rgba())

    if ALPHA:

        frames.append(frame_rgba.copy())

    else:

        frames.append(frame_rgba[:, :, :3].copy())







plt.close(fig)











out_file = export_animation(



    frames=frames,



    out_dir=OUT_DIR,



    animation_name="warning_system_v1",



    output_format=OUTPUT_FORMAT,



    fps=FPS,



    alpha=ALPHA,



)







print(f"Saved: {out_file}")



print(f"Frames: {len(frames)}")



print(f"Size: {out_file.stat().st_size / 1024:.1f} KB")



ffmpeg version 7.1.1 Copyright (c) 2000-2025 the FFmpeg developers
  built with clang version 18.1.8
  configuration: --prefix=/Users/mloktionov/anaconda3/envs/astro-ai --cc=arm64-apple-darwin20.0.0-clang --cxx=arm64-apple-darwin20.0.0-clang++ --nm=arm64-apple-darwin20.0.0-nm --ar=arm64-apple-darwin20.0.0-ar --disable-doc --enable-openssl --enable-demuxer=dash --enable-hardcoded-tables --enable-libfreetype --enable-libharfbuzz --enable-libfontconfig --enable-libopenh264 --enable-libdav1d --enable-cross-compile --arch=arm64 --target-os=darwin --cross-prefix=arm64-apple-darwin20.0.0- --host-cc=/Users/runner/miniforge3/conda-bld/ffmpeg_1748704173249/_build_env/bin/x86_64-apple-darwin13.4.0-clang --enable-neon --disable-gnutls --enable-libvpx --enable-libass --enable-pthreads --enable-libopenvino --enable-gpl --enable-libx264 --enable-libx265 --enable-libmp3lame --enable-libaom --enable-libsvtav1 --enable-libxml2 --enable-pic --enable-shared --disable-static --enable-version3 --enable-zlib

Saved: animations/HUD/Warning_System/warning_system_v1.webm
Frames: 144
Size: 782.4 KB


[out#0/webm @ 0x13c105a40] video:251KiB audio:0KiB subtitle:0KiB other streams:0KiB global headers:0KiB muxing overhead: 212.022682%
frame=  144 fps= 25 q=32.0 Lsize=     782KiB time=00:00:06.00 bitrate=1068.3kbits/s speed=1.06x    


# Planetary Scanner v1

In [17]:
# Planetary Scanner v1 — animated transparent planetary scan HUD















# Output: media-site/animations/HUD/Planetary_Scanner/planetary_scanner_v1.webm































from pathlib import Path































import numpy as np















import matplotlib.pyplot as plt















from matplotlib.patches import Circle, Arc















from matplotlib.lines import Line2D































from vizlib.animation_export import export_animation















































OUTPUT_FORMAT = "webm"   # webm | mp4 | gif















ALPHA = False































OUT_DIR = Path("media-site/animations/HUD/Planetary_Scanner")















panel_x = 0.65







panel_y = 0.75







panel_w = 5.2







panel_h = 7.5































W, H = 16, 9















DPI = 120















FPS = 24















DURATION = 6















FRAMES = FPS * DURATION































COL = "#35f6ff"















COL_DIM = "#1b7f88"















COL_WARN = "#ff4d4d"















COL_SOFT = "#9ffcff"















COL_AMBER = "#ffb000"































LOOPS_SCAN = 2















LOOPS_PULSE = 4















LOOPS_CONTOUR = 3















LOOPS_MARKERS = 5















LOOPS_DATA = 3















































fig, ax = plt.subplots(figsize=(W, H), dpi=DPI)































if ALPHA:















    fig.patch.set_facecolor((0, 0, 0, 0))















    fig.patch.set_alpha(0.0)















    ax.set_facecolor((0, 0, 0, 0))















else:















    fig.patch.set_facecolor("black")















    ax.set_facecolor("black")































ax.set_xlim(-16, 16)















ax.set_ylim(-9, 9)















ax.set_aspect("equal")















ax.axis("off")















































CENTER = (-2.2, 0.0)















R = 4.1















































# Planet outline















planet_edge = Circle(















    CENTER,















    R,















    fill=False,















    ec=COL,















    lw=1.4,















    alpha=0.75,















)















ax.add_patch(planet_edge)































planet_fill = Circle(















    CENTER,















    R,















    fill=True,















    color=COL,















    alpha=0.025,















)















ax.add_patch(planet_fill)















































# Latitude lines















lat_lines = []















for y_frac in [-0.75, -0.5, -0.25, 0.0, 0.25, 0.5, 0.75]:















    y = CENTER[1] + R * y_frac















    half_width = R * np.sqrt(max(0.0, 1.0 - y_frac**2))































    line = Line2D(















        [CENTER[0] - half_width, CENTER[0] + half_width],















        [y, y],















        lw=0.65,















        color=COL_DIM,















        alpha=0.23,















    )















    ax.add_line(line)















    lat_lines.append(line)















































# Longitude arcs















lon_arcs = []















for width_scale in [0.25, 0.45, 0.65, 0.85]:















    arc = Arc(















        CENTER,















        2 * R * width_scale,















        2 * R,















        angle=0,















        theta1=90,















        theta2=270,















        lw=0.55,















        color=COL_DIM,















        alpha=0.22,















    )















    ax.add_patch(arc)















    lon_arcs.append(arc)































    arc2 = Arc(















        CENTER,















        2 * R * width_scale,















        2 * R,















        angle=0,















        theta1=-90,















        theta2=90,















        lw=0.55,















        color=COL_DIM,















        alpha=0.22,















    )















    ax.add_patch(arc2)















    lon_arcs.append(arc2)















































# Scanning meridian















scan_arc = Arc(















    CENTER,















    2 * R,















    2 * R,















    angle=0,















    theta1=70,















    theta2=110,















    lw=2.2,















    color=COL_SOFT,















    alpha=0.9,















)















ax.add_patch(scan_arc)































scan_line = Line2D([], [], lw=1.0, color=COL_SOFT, alpha=0.8)















ax.add_line(scan_line)















































# Contour lines on planet















rng = np.random.default_rng(12)















contours = []































for j in range(7):















    pts = []















    base_y = -2.6 + j * 0.85































    xs = np.linspace(CENTER[0] - 3.0, CENTER[0] + 3.0, 90)































    for x in xs:















        dx = x - CENTER[0]















        y = CENTER[1] + base_y + 0.18 * np.sin(1.8 * dx + j * 0.9)















        if (x - CENTER[0]) ** 2 + (y - CENTER[1]) ** 2 <= (R * 0.92) ** 2:















            pts.append((x, y))































    if len(pts) > 2:















        line, = ax.plot(















            [p[0] for p in pts],















            [p[1] for p in pts],















            lw=0.75,















            color=COL,















            alpha=0.2,















        )















        contours.append((line, j))















































# Surface markers















marker_defs = [















    (-4.1, 1.3, "A1"),















    (-1.0, 2.2, "B4"),















    (-3.2, -1.7, "C2"),















    (0.6, -0.8, "D7"),















]































markers = []















marker_labels = []































for x, y, label in marker_defs:















    dot = Circle((x, y), 0.08, fill=True, color=COL_WARN, alpha=0.55)















    halo = Circle((x, y), 0.28, fill=False, ec=COL_WARN, lw=0.8, alpha=0.25)































    ax.add_patch(halo)















    ax.add_patch(dot)































    txt = ax.text(















        x + 0.17,















        y + 0.1,















        label,















        color=COL_WARN,















        fontsize=7,















        family="monospace",















        alpha=0.55,















    )































    markers.append((dot, halo))















    marker_labels.append(txt)































































































txt_title = ax.text(















    panel_x + 0.35,















    panel_y + panel_h - 0.65,















    "PLANETARY SCANNER // SURFACE MODEL",















    color=COL,















    fontsize=12,















    family="monospace",















    alpha=0.95,















)































txt_state = ax.text(















    panel_x + 0.35,















    panel_y + panel_h - 1.2,















    "",















    color=COL_DIM,















    fontsize=9,















    family="monospace",















    alpha=0.85,















)































txt_geo = ax.text(















    panel_x + 0.35,















    panel_y + panel_h - 2.05,















    "",















    color=COL_SOFT,















    fontsize=9,















    family="monospace",















    alpha=0.85,















)































txt_atmo = ax.text(















    panel_x + 0.35,















    panel_y + panel_h - 2.65,















    "",















    color=COL_SOFT,















    fontsize=9,















    family="monospace",















    alpha=0.85,















)































txt_thermal = ax.text(















    panel_x + 0.35,















    panel_y + panel_h - 3.25,















    "",















    color=COL_SOFT,















    fontsize=9,















    family="monospace",















    alpha=0.85,















)































txt_alert = ax.text(















    panel_x + 0.35,















    panel_y + 0.55,















    "ANOMALY INDEX: LOW",















    color=COL_AMBER,















    fontsize=10,















    family="monospace",















    alpha=0.75,















)















































# Small data bars















bars = []















for i in range(34):















    x = panel_x + 0.35 + i * 0.27















    bar = Line2D(















        [x, x],















        [panel_y + 1.25, panel_y + 1.7],















        lw=1.4,















        color=COL,















        alpha=0.22,















    )















    ax.add_line(bar)















    bars.append(bar)















































def update(frame_idx: int) -> None:















    t = frame_idx / FRAMES















    tau = 2 * np.pi































    pulse = 0.5 + 0.5 * np.sin(tau * LOOPS_PULSE * t)















    scan_phase = 360 * LOOPS_SCAN * t































    planet_edge.set_alpha(0.45 + 0.35 * pulse)















    planet_fill.set_alpha(0.015 + 0.035 * pulse)































































    for i, arc in enumerate(lon_arcs):















        phase = tau * LOOPS_CONTOUR * t + i * 0.28















        arc.set_alpha(0.08 + 0.23 * (0.5 + 0.5 * np.sin(phase)))































    # Rotating scanner















    scan_arc.angle = scan_phase















    scan_arc.set_alpha(0.35 + 0.55 * pulse)































    a = np.deg2rad(scan_phase)















    scan_line.set_data(















        [CENTER[0], CENTER[0] + R * np.cos(a)],















        [CENTER[1], CENTER[1] + R * np.sin(a)],















    )















    scan_line.set_alpha(0.18 + 0.45 * pulse)































    # Surface contours















    for line, j in contours:















        phase = tau * LOOPS_CONTOUR * t + j * 0.7















        line.set_alpha(0.08 + 0.35 * (0.5 + 0.5 * np.sin(phase)))































    # Markers















    for i, ((dot, halo), label) in enumerate(zip(markers, marker_labels)):















        phase = tau * LOOPS_MARKERS * t + i * 0.9















        k = 0.5 + 0.5 * np.sin(phase)































        dot.set_alpha(0.25 + 0.65 * k)















        halo.set_alpha(0.1 + 0.45 * k)















        halo.set_radius(0.22 + 0.22 * k)















        label.set_alpha(0.25 + 0.55 * k)















































    mineral = 62 + int(13 * np.sin(tau * LOOPS_DATA * t))















    atmo = 18 + int(7 * np.cos(tau * LOOPS_DATA * t))















    temp = 142 + int(9 * np.sin(tau * LOOPS_DATA * t + 1.1))















    scan_pct = int(50 + 49 * (0.5 + 0.5 * np.sin(tau * LOOPS_SCAN * t - np.pi / 2)))































    txt_state.set_text(f"SCAN COMPLETION {scan_pct:02d}%  ROTATION LOCKED")















    txt_geo.set_text(f"MINERAL REFLECTANCE {mineral:02d}%  CRUSTAL VARIANCE 0.42")















    txt_atmo.set_text(f"ATMOSPHERE TRACE {atmo:02d}%  PRESSURE MODEL WEAK")















    txt_thermal.set_text(f"THERMAL MAP {temp:03d} K  NIGHT-SIDE GRADIENT DETECTED")































    txt_title.set_alpha(0.7 + 0.25 * pulse)















    txt_alert.set_alpha(0.35 + 0.45 * pulse)































    # Bars















    for i, bar in enumerate(bars):















        phase = tau * LOOPS_DATA * t + i * 0.31















        k = 0.5 + 0.5 * np.sin(phase)















        x = panel_x + 0.35 + i * 0.27































        bar.set_data(















            [x, x],















            [panel_y + 1.25, panel_y + 1.25 + 0.75 * k],















        )















        bar.set_alpha(0.08 + 0.55 * k)















































frames = []































for frame_idx in range(FRAMES):















    update(frame_idx)















    fig.canvas.draw()































    frame_rgba = np.asarray(fig.canvas.renderer.buffer_rgba())































    frames.append(frame_rgba[:, :, :3].copy())































plt.close(fig)















































out_file = export_animation(















    frames=frames,















    out_dir=OUT_DIR,















    animation_name="planetary_scanner_v1",















    output_format=OUTPUT_FORMAT,















    fps=FPS,















    alpha=ALPHA,















)































print(f"Saved: {out_file}")















print(f"Frames: {len(frames)}")















print(f"Size: {out_file.stat().st_size / 1024:.1f} KB")









ffmpeg version 7.1.1 Copyright (c) 2000-2025 the FFmpeg developers
  built with clang version 18.1.8
  configuration: --prefix=/Users/mloktionov/anaconda3/envs/astro-ai --cc=arm64-apple-darwin20.0.0-clang --cxx=arm64-apple-darwin20.0.0-clang++ --nm=arm64-apple-darwin20.0.0-nm --ar=arm64-apple-darwin20.0.0-ar --disable-doc --enable-openssl --enable-demuxer=dash --enable-hardcoded-tables --enable-libfreetype --enable-libharfbuzz --enable-libfontconfig --enable-libopenh264 --enable-libdav1d --enable-cross-compile --arch=arm64 --target-os=darwin --cross-prefix=arm64-apple-darwin20.0.0- --host-cc=/Users/runner/miniforge3/conda-bld/ffmpeg_1748704173249/_build_env/bin/x86_64-apple-darwin13.4.0-clang --enable-neon --disable-gnutls --enable-libvpx --enable-libass --enable-pthreads --enable-libopenvino --enable-gpl --enable-libx264 --enable-libx265 --enable-libmp3lame --enable-libaom --enable-libsvtav1 --enable-libxml2 --enable-pic --enable-shared --disable-static --enable-version3 --enable-zlib

Saved: animations/HUD/Planetary_Scanner/planetary_scanner_v1.webm
Frames: 144
Size: 714.4 KB


[out#0/webm @ 0x14c6044d0] video:290KiB audio:0KiB subtitle:0KiB other streams:0KiB global headers:0KiB muxing overhead: 146.399677%
frame=  144 fps= 29 q=32.0 Lsize=     714KiB time=00:00:06.00 bitrate= 975.4kbits/s speed= 1.2x    


In [16]:
# Planetary Scanner v1 — vertical layout







# Output: media-site/animations/HUD/Planetary_Scanner/planetary_scanner_v1_vertical.webm















from pathlib import Path















import numpy as np







import matplotlib.pyplot as plt







from matplotlib.patches import Circle, Arc







from matplotlib.lines import Line2D















from vizlib.animation_export import export_animation























OUTPUT_FORMAT = "webm"







ALPHA = False















OUT_DIR = Path("media-site/animations/HUD/Planetary_Scanner")







panel_x = 0.65



panel_y = 0.75



panel_w = 5.2



panel_h = 7.5















W, H = 9, 16







DPI = 120







FPS = 24







DURATION = 6







FRAMES = FPS * DURATION















COL = "#35f6ff"







COL_DIM = "#1b7f88"







COL_WARN = "#ff4d4d"







COL_SOFT = "#9ffcff"







COL_AMBER = "#ffb000"















LOOPS_SCAN = 2







LOOPS_PULSE = 4







LOOPS_CONTOUR = 3







LOOPS_MARKERS = 5







LOOPS_DATA = 3























fig, ax = plt.subplots(figsize=(W, H), dpi=DPI)















if ALPHA:







    fig.patch.set_facecolor((0, 0, 0, 0))







    fig.patch.set_alpha(0.0)







    ax.set_facecolor((0, 0, 0, 0))







else:







    fig.patch.set_facecolor("black")







    ax.set_facecolor("black")















ax.set_xlim(-9, 9)







ax.set_ylim(-16, 16)







ax.set_aspect("equal")







ax.axis("off")























CENTER = (0.0, 4.0)







R = 5.6























# Planet body







planet_edge = Circle(







    CENTER,







    R,







    fill=False,







    ec=COL,







    lw=1.5,







    alpha=0.75,







)







ax.add_patch(planet_edge)















planet_fill = Circle(







    CENTER,







    R,







    fill=True,







    color=COL,







    alpha=0.025,







)







ax.add_patch(planet_fill)























# Latitude lines







lat_lines = []















for y_frac in [-0.75, -0.5, -0.25, 0.0, 0.25, 0.5, 0.75]:







    y = CENTER[1] + R * y_frac







    half_width = R * np.sqrt(max(0.0, 1.0 - y_frac**2))















    line = Line2D(







        [CENTER[0] - half_width, CENTER[0] + half_width],







        [y, y],







        lw=0.65,







        color=COL_DIM,







        alpha=0.23,







    )















    ax.add_line(line)







    lat_lines.append(line)























# Longitude arcs







lon_arcs = []















for width_scale in [0.25, 0.45, 0.65, 0.85]:







    arc1 = Arc(







        CENTER,







        2 * R * width_scale,







        2 * R,







        angle=0,







        theta1=90,







        theta2=270,







        lw=0.55,







        color=COL_DIM,







        alpha=0.22,







    )















    arc2 = Arc(







        CENTER,







        2 * R * width_scale,







        2 * R,







        angle=0,







        theta1=-90,







        theta2=90,







        lw=0.55,







        color=COL_DIM,







        alpha=0.22,







    )















    ax.add_patch(arc1)







    ax.add_patch(arc2)















    lon_arcs.extend([arc1, arc2])























# Rotating scanner







scan_arc = Arc(







    CENTER,







    2 * R,







    2 * R,







    angle=0,







    theta1=70,







    theta2=110,







    lw=2.2,







    color=COL_SOFT,







    alpha=0.9,







)















ax.add_patch(scan_arc)















scan_line = Line2D([], [], lw=1.0, color=COL_SOFT, alpha=0.8)







ax.add_line(scan_line)























# Contours







contours = []















for j in range(9):







    pts = []















    base_y = -4.1 + j * 1.0















    xs = np.linspace(CENTER[0] - 4.4, CENTER[0] + 4.4, 120)















    for x in xs:







        dx = x - CENTER[0]















        y = CENTER[1] + base_y + 0.25 * np.sin(1.7 * dx + j * 0.9)















        if (x - CENTER[0])**2 + (y - CENTER[1])**2 <= (R * 0.93)**2:







            pts.append((x, y))















    if len(pts) > 2:







        line, = ax.plot(







            [p[0] for p in pts],







            [p[1] for p in pts],







            lw=0.75,







            color=COL,







            alpha=0.18,







        )















        contours.append((line, j))























# Surface markers







marker_defs = [







    (-3.9, 6.0, "A1"),







    (2.8, 7.3, "B4"),







    (-2.2, 1.2, "C2"),







    (3.4, 2.0, "D7"),







]















markers = []







marker_labels = []















for x, y, label in marker_defs:







    dot = Circle((x, y), 0.09, fill=True, color=COL_WARN, alpha=0.55)







    halo = Circle((x, y), 0.32, fill=False, ec=COL_WARN, lw=0.8, alpha=0.25)















    ax.add_patch(halo)







    ax.add_patch(dot)















    txt = ax.text(







        x + 0.2,







        y + 0.15,







        label,







        color=COL_WARN,







        fontsize=7,







        family="monospace",







        alpha=0.55,







    )















    markers.append((dot, halo))







    marker_labels.append(txt)























# Telemetry panel bottom







txt_title = ax.text(







    -7.5,







    -6.4,







    "PLANETARY SCANNER",







    color=COL,







    fontsize=12,







    family="monospace",







    alpha=0.95,







)















txt_state = ax.text(







    -7.5,







    -7.2,







    "",







    color=COL_DIM,







    fontsize=8.5,







    family="monospace",







    alpha=0.85,







)















txt_geo = ax.text(







    -7.5,







    -8.4,







    "",







    color=COL_SOFT,







    fontsize=8.5,







    family="monospace",







    alpha=0.85,







)















txt_atmo = ax.text(







    -7.5,







    -9.3,







    "",







    color=COL_SOFT,







    fontsize=8.5,







    family="monospace",







    alpha=0.85,







)















txt_thermal = ax.text(







    -7.5,







    -10.2,







    "",







    color=COL_SOFT,







    fontsize=8.5,







    family="monospace",







    alpha=0.85,







)















txt_alert = ax.text(







    -7.5,







    -12.0,







    "ANOMALY INDEX: LOW",







    color=COL_AMBER,







    fontsize=9,







    family="monospace",







    alpha=0.75,







)























# Bottom bars







bars = []















for i in range(22):







    x = -7.4 + i * 0.62















    bar = Line2D(







        [x, x],







        [-14.0, -13.2],







        lw=2.0,







        color=COL,







        alpha=0.25,







    )















    ax.add_line(bar)







    bars.append(bar)























def update(frame_idx: int) -> None:







    t = frame_idx / FRAMES







    tau = 2 * np.pi















    pulse = 0.5 + 0.5 * np.sin(tau * LOOPS_PULSE * t)







    scan_phase = 360 * LOOPS_SCAN * t















    planet_edge.set_alpha(0.45 + 0.35 * pulse)







    planet_fill.set_alpha(0.015 + 0.04 * pulse)















    for i, line in enumerate(lat_lines):







        phase = tau * LOOPS_CONTOUR * t + i * 0.35







        line.set_alpha(0.08 + 0.25 * (0.5 + 0.5 * np.sin(phase)))















    for i, arc in enumerate(lon_arcs):







        phase = tau * LOOPS_CONTOUR * t + i * 0.28







        arc.set_alpha(0.08 + 0.23 * (0.5 + 0.5 * np.sin(phase)))















    # Rotating scan







    scan_arc.angle = scan_phase







    scan_arc.set_alpha(0.35 + 0.55 * pulse)















    a = np.deg2rad(scan_phase)















    scan_line.set_data(







        [CENTER[0], CENTER[0] + R * np.cos(a)],







        [CENTER[1], CENTER[1] + R * np.sin(a)],







    )















    scan_line.set_alpha(0.18 + 0.45 * pulse)















    # Contours







    for line, j in contours:







        phase = tau * LOOPS_CONTOUR * t + j * 0.7







        line.set_alpha(0.08 + 0.35 * (0.5 + 0.5 * np.sin(phase)))















    # Markers







    for i, ((dot, halo), label) in enumerate(zip(markers, marker_labels)):







        phase = tau * LOOPS_MARKERS * t + i * 0.9







        k = 0.5 + 0.5 * np.sin(phase)















        dot.set_alpha(0.25 + 0.65 * k)















        halo.set_alpha(0.1 + 0.45 * k)







        halo.set_radius(0.22 + 0.25 * k)















        label.set_alpha(0.25 + 0.55 * k)















    # Telemetry







    mineral = 62 + int(13 * np.sin(tau * LOOPS_DATA * t))







    atmo = 18 + int(7 * np.cos(tau * LOOPS_DATA * t))







    temp = 142 + int(9 * np.sin(tau * LOOPS_DATA * t + 1.1))















    scan_pct = int(







        50 + 49 * (







            0.5 + 0.5 * np.sin(







                tau * LOOPS_SCAN * t - np.pi / 2







            )







        )







    )















    txt_state.set_text(







        f"SCAN {scan_pct:02d}%  ROTATION LOCKED"







    )















    txt_geo.set_text(







        f"MINERAL REFLECTANCE {mineral:02d}%"







    )















    txt_atmo.set_text(







        f"ATMOSPHERIC TRACE {atmo:02d}%"







    )















    txt_thermal.set_text(







        f"THERMAL MAP {temp:03d} K"







    )















    txt_title.set_alpha(0.7 + 0.25 * pulse)







    txt_alert.set_alpha(0.35 + 0.45 * pulse)















    # Bars







    for i, bar in enumerate(bars):







        phase = tau * LOOPS_DATA * t + i * 0.31















        k = 0.5 + 0.5 * np.sin(phase)















        x = -7.4 + i * 0.62















        bar.set_data(







            [x, x],







            [-14.0, -14.0 + 1.6 * k],







        )















        bar.set_alpha(0.08 + 0.55 * k)























frames = []















for frame_idx in range(FRAMES):







    update(frame_idx)















    fig.canvas.draw()















    frame_rgba = np.asarray(fig.canvas.renderer.buffer_rgba())



    if ALPHA:



        frames.append(frame_rgba.copy())



    else:



        frames.append(frame_rgba[:, :, :3].copy())















plt.close(fig)























out_file = export_animation(







    frames=frames,







    out_dir=OUT_DIR,







    animation_name="planetary_scanner_v1_vertical",







    output_format=OUTPUT_FORMAT,







    fps=FPS,







    alpha=ALPHA,







)















print(f"Saved: {out_file}")







print(f"Frames: {len(frames)}")







print(f"Size: {out_file.stat().st_size / 1024:.1f} KB")





ffmpeg version 7.1.1 Copyright (c) 2000-2025 the FFmpeg developers
  built with clang version 18.1.8
  configuration: --prefix=/Users/mloktionov/anaconda3/envs/astro-ai --cc=arm64-apple-darwin20.0.0-clang --cxx=arm64-apple-darwin20.0.0-clang++ --nm=arm64-apple-darwin20.0.0-nm --ar=arm64-apple-darwin20.0.0-ar --disable-doc --enable-openssl --enable-demuxer=dash --enable-hardcoded-tables --enable-libfreetype --enable-libharfbuzz --enable-libfontconfig --enable-libopenh264 --enable-libdav1d --enable-cross-compile --arch=arm64 --target-os=darwin --cross-prefix=arm64-apple-darwin20.0.0- --host-cc=/Users/runner/miniforge3/conda-bld/ffmpeg_1748704173249/_build_env/bin/x86_64-apple-darwin13.4.0-clang --enable-neon --disable-gnutls --enable-libvpx --enable-libass --enable-pthreads --enable-libopenvino --enable-gpl --enable-libx264 --enable-libx265 --enable-libmp3lame --enable-libaom --enable-libsvtav1 --enable-libxml2 --enable-pic --enable-shared --disable-static --enable-version3 --enable-zlib

Saved: animations/HUD/Planetary_Scanner/planetary_scanner_v1_vertical.webm
Frames: 144
Size: 875.6 KB


In [18]:
# Deep Space Navigation HUD v1 — animated transparent route overlay



# Output: media-site/animations/HUD/Deep_Space_Navigation/deep_space_navigation_v1.webm







from pathlib import Path







import numpy as np



import matplotlib.pyplot as plt



from matplotlib.patches import Circle



from matplotlib.lines import Line2D







from vizlib.animation_export import export_animation











OUTPUT_FORMAT = "webm"   # webm | mp4 | gif



ALPHA = False







OUT_DIR = Path("media-site/animations/HUD/Deep_Space_Navigation")







W, H = 16, 9



DPI = 120



FPS = 24



DURATION = 6



FRAMES = FPS * DURATION







COL = "#35f6ff"



COL_DIM = "#1b7f88"



COL_SOFT = "#9ffcff"



COL_WARN = "#ff4d4d"



COL_AMBER = "#ffb000"







LOOPS_ROUTE = 1



LOOPS_PULSE = 4



LOOPS_STARS = 3



LOOPS_DATA = 3



LOOPS_MARKERS = 5











fig, ax = plt.subplots(figsize=(W, H), dpi=DPI)







if ALPHA:



    fig.patch.set_facecolor((0, 0, 0, 0))



    fig.patch.set_alpha(0.0)



    ax.set_facecolor((0, 0, 0, 0))



else:



    fig.patch.set_facecolor("black")



    ax.set_facecolor("black")







ax.set_xlim(-16, 16)



ax.set_ylim(-9, 9)



ax.set_aspect("equal")



ax.axis("off")











rng = np.random.default_rng(77)











# Background navigation stars



stars = []



star_defs = []







for _ in range(90):



    x = rng.uniform(-14.8, 14.8)



    y = rng.uniform(-7.6, 7.6)



    size = rng.uniform(0.8, 2.2)



    phase = rng.uniform(0, 2 * np.pi)







    star, = ax.plot([x], [y], "o", ms=size, color=COL_SOFT, alpha=0.0)



    stars.append(star)



    star_defs.append((x, y, size, phase))











# Route waypoints



waypoints = np.array([



    [-12.2, -3.2],



    [-8.4, -1.1],



    [-4.2, 1.9],



    [0.8, 0.7],



    [5.4, 2.8],



    [11.8, 0.2],



])







route_segments = []







for i in range(len(waypoints) - 1):



    x0, y0 = waypoints[i]



    x1, y1 = waypoints[i + 1]







    line = Line2D(



        [x0, x1],



        [y0, y1],



        lw=1.0,



        color=COL_DIM,



        alpha=0.35,



    )



    ax.add_line(line)



    route_segments.append(line)











# Route glow/progress segments



active_segments = []







for i in range(len(waypoints) - 1):



    line = Line2D(



        [],



        [],



        lw=2.0,



        color=COL,



        alpha=0.0,



    )



    ax.add_line(line)



    active_segments.append(line)











# Waypoint markers



waypoint_markers = []



waypoint_labels = []







for i, (x, y) in enumerate(waypoints):



    dot = Circle((x, y), 0.09, fill=True, color=COL, alpha=0.65)



    halo = Circle((x, y), 0.32, fill=False, ec=COL, lw=0.8, alpha=0.25)







    ax.add_patch(halo)



    ax.add_patch(dot)







    label = ax.text(



        x + 0.25,



        y + 0.18,



        f"NAV-{i + 1}",



        color=COL_DIM,



        fontsize=7,



        family="monospace",



        alpha=0.65,



    )







    waypoint_markers.append((dot, halo))



    waypoint_labels.append(label)











# Target system marker



target_x, target_y = waypoints[-1]



target_ring_1 = Circle((target_x, target_y), 0.55, fill=False, ec=COL_WARN, lw=1.2, alpha=0.8)



target_ring_2 = Circle((target_x, target_y), 0.9, fill=False, ec=COL_WARN, lw=0.8, alpha=0.35)



target_dot = Circle((target_x, target_y), 0.08, fill=True, color=COL_WARN, alpha=0.9)







ax.add_patch(target_ring_1)



ax.add_patch(target_ring_2)



ax.add_patch(target_dot)











# Moving ship marker



ship_marker = Circle((waypoints[0][0], waypoints[0][1]), 0.13, fill=True, color=COL_AMBER, alpha=0.95)



ship_halo = Circle((waypoints[0][0], waypoints[0][1]), 0.45, fill=False, ec=COL_AMBER, lw=0.9, alpha=0.5)



ax.add_patch(ship_halo)



ax.add_patch(ship_marker)











# Jump vector projection



vector_line = Line2D([], [], lw=1.1, color=COL_AMBER, alpha=0.55)



ax.add_line(vector_line)







vector_head = Circle((0, 0), 0.08, fill=True, color=COL_AMBER, alpha=0.0)



ax.add_patch(vector_head)











# Coordinate crosshair around target



target_ticks = []







for dx0, dy0, dx1, dy1 in [



    (-1.35, 0, -0.72, 0),



    (0.72, 0, 1.35, 0),



    (0, -1.35, 0, -0.72),



    (0, 0.72, 0, 1.35),



]:



    tick = Line2D(



        [target_x + dx0, target_x + dx1],



        [target_y + dy0, target_y + dy1],



        lw=1.1,



        color=COL_WARN,



        alpha=0.55,



    )



    ax.add_line(tick)



    target_ticks.append(tick)











# Telemetry labels



txt_title = ax.text(



    -14.7,



    7.45,



    "DEEP SPACE NAVIGATION // JUMP VECTOR SOLUTION",



    color=COL,



    fontsize=12,



    family="monospace",



    alpha=0.95,



)







txt_state = ax.text(



    -14.7,



    6.9,



    "",



    color=COL_DIM,



    fontsize=9,



    family="monospace",



    alpha=0.85,



)







txt_target = ax.text(



    5.6,



    -7.55,



    "",



    color=COL_SOFT,



    fontsize=9,



    family="monospace",



    alpha=0.85,



)







txt_warning = ax.text(



    -1.8,



    -7.45,



    "ROUTE LOCK",



    color=COL_AMBER,



    fontsize=12,



    family="monospace",



    alpha=0.75,



)











def interpolate_route(points: np.ndarray, progress: float) -> tuple[np.ndarray, int, float]:



    progress = progress % 1.0







    seg_count = len(points) - 1



    scaled = progress * seg_count



    seg_idx = min(int(scaled), seg_count - 1)



    local_t = scaled - seg_idx







    p0 = points[seg_idx]



    p1 = points[seg_idx + 1]







    pos = p0 * (1 - local_t) + p1 * local_t







    return pos, seg_idx, local_t











def update(frame_idx: int) -> None:



    t = frame_idx / FRAMES



    tau = 2 * np.pi







    pulse = 0.5 + 0.5 * np.sin(tau * LOOPS_PULSE * t)



    route_progress = (LOOPS_ROUTE * t) % 1.0







    # Star flicker



    for i, star in enumerate(stars):



        _, _, _, phase = star_defs[i]



        k = 0.5 + 0.5 * np.sin(tau * LOOPS_STARS * t + phase)



        star.set_alpha(0.08 + 0.45 * k)







    # Base route segments



    for i, line in enumerate(route_segments):



        phase = tau * LOOPS_DATA * t + i * 0.65



        line.set_alpha(0.12 + 0.28 * (0.5 + 0.5 * np.sin(phase)))







    # Active route trace



    seg_count = len(waypoints) - 1



    scaled = route_progress * seg_count







    for i, line in enumerate(active_segments):



        if i < int(scaled):



            p0 = waypoints[i]



            p1 = waypoints[i + 1]



            line.set_data([p0[0], p1[0]], [p0[1], p1[1]])



            line.set_alpha(0.55 + 0.35 * pulse)



        elif i == int(scaled):



            p0 = waypoints[i]



            p1 = waypoints[i + 1]



            local_t = scaled - i



            p = p0 * (1 - local_t) + p1 * local_t



            line.set_data([p0[0], p[0]], [p0[1], p[1]])



            line.set_alpha(0.55 + 0.35 * pulse)



        else:



            line.set_data([], [])



            line.set_alpha(0.0)







    # Ship marker



    ship_pos, seg_idx, local_t = interpolate_route(waypoints, route_progress)



    sx, sy = ship_pos







    ship_marker.center = (sx, sy)



    ship_halo.center = (sx, sy)



    ship_marker.set_alpha(0.65 + 0.3 * pulse)



    ship_halo.set_alpha(0.2 + 0.45 * pulse)



    ship_halo.set_radius(0.32 + 0.22 * pulse)







    # Vector projection to target



    vector_line.set_data([sx, target_x], [sy, target_y])



    vector_line.set_alpha(0.18 + 0.42 * pulse)







    vx = sx + 0.68 * (target_x - sx)



    vy = sy + 0.68 * (target_y - sy)



    vector_head.center = (vx, vy)



    vector_head.set_alpha(0.35 + 0.45 * pulse)







    # Waypoint markers



    for i, ((dot, halo), label) in enumerate(zip(waypoint_markers, waypoint_labels)):



        phase = tau * LOOPS_MARKERS * t + i * 0.75



        k = 0.5 + 0.5 * np.sin(phase)







        dot.set_alpha(0.25 + 0.55 * k)



        halo.set_alpha(0.08 + 0.35 * k)



        halo.set_radius(0.22 + 0.22 * k)



        label.set_alpha(0.25 + 0.45 * k)







    # Target lock animation



    target_ring_1.set_alpha(0.35 + 0.55 * pulse)



    target_ring_1.set_radius(0.44 + 0.16 * pulse)







    target_ring_2.set_alpha(0.15 + 0.35 * pulse)



    target_ring_2.set_radius(0.72 + 0.28 * pulse)







    target_dot.set_alpha(0.65 + 0.3 * pulse)







    for i, tick in enumerate(target_ticks):



        phase = tau * LOOPS_PULSE * t + i * 0.6



        tick.set_alpha(0.2 + 0.5 * (0.5 + 0.5 * np.sin(phase)))







    # Telemetry



    remaining = np.linalg.norm(np.array([target_x - sx, target_y - sy]))



    phase_deg = 360 * route_progress



    jump_stability = 92 + int(6 * np.sin(tau * LOOPS_DATA * t))







    txt_state.set_text(



        f"ROUTE PHASE {phase_deg:06.2f} DEG  SEGMENT {seg_idx + 1:02d}  STABILITY {jump_stability:02d}%"



    )







    txt_target.set_text(



        f"TARGET SYSTEM: FOMALHAUT-BRIDGE  DIST {remaining:05.2f} LY  VECTOR CLEAN"



    )







    txt_title.set_alpha(0.7 + 0.25 * pulse)



    txt_warning.set_alpha(0.35 + 0.55 * pulse)











frames = []







for frame_idx in range(FRAMES):



    update(frame_idx)



    fig.canvas.draw()







    frame_rgba = np.asarray(fig.canvas.renderer.buffer_rgba())

    if ALPHA:

        frames.append(frame_rgba.copy())

    else:

        frames.append(frame_rgba[:, :, :3].copy())







plt.close(fig)











out_file = export_animation(



    frames=frames,



    out_dir=OUT_DIR,



    animation_name="deep_space_navigation_v1",



    output_format=OUTPUT_FORMAT,



    fps=FPS,



    alpha=ALPHA,



)







print(f"Saved: {out_file}")



print(f"Frames: {len(frames)}")



print(f"Size: {out_file.stat().st_size / 1024:.1f} KB")



ffmpeg version 7.1.1 Copyright (c) 2000-2025 the FFmpeg developers
  built with clang version 18.1.8
  configuration: --prefix=/Users/mloktionov/anaconda3/envs/astro-ai --cc=arm64-apple-darwin20.0.0-clang --cxx=arm64-apple-darwin20.0.0-clang++ --nm=arm64-apple-darwin20.0.0-nm --ar=arm64-apple-darwin20.0.0-ar --disable-doc --enable-openssl --enable-demuxer=dash --enable-hardcoded-tables --enable-libfreetype --enable-libharfbuzz --enable-libfontconfig --enable-libopenh264 --enable-libdav1d --enable-cross-compile --arch=arm64 --target-os=darwin --cross-prefix=arm64-apple-darwin20.0.0- --host-cc=/Users/runner/miniforge3/conda-bld/ffmpeg_1748704173249/_build_env/bin/x86_64-apple-darwin13.4.0-clang --enable-neon --disable-gnutls --enable-libvpx --enable-libass --enable-pthreads --enable-libopenvino --enable-gpl --enable-libx264 --enable-libx265 --enable-libmp3lame --enable-libaom --enable-libsvtav1 --enable-libxml2 --enable-pic --enable-shared --disable-static --enable-version3 --enable-zlib

Saved: animations/HUD/Deep_Space_Navigation/deep_space_navigation_v1.webm
Frames: 144
Size: 792.7 KB


[out#0/webm @ 0x13eb04cf0] video:402KiB audio:0KiB subtitle:0KiB other streams:0KiB global headers:0KiB muxing overhead: 97.120106%
frame=  144 fps= 30 q=32.0 Lsize=     793KiB time=00:00:06.00 bitrate=1082.3kbits/s speed=1.24x    


# Holographic Grid / Starfield Overlay v1

In [19]:
# Holographic Grid Starfield v1 — animated transparent grid/star overlay



# Output: media-site/animations/HUD/Holographic_Grid/holographic_grid_starfield_v1.webm







from pathlib import Path







import numpy as np



import matplotlib.pyplot as plt



from matplotlib.patches import Circle, Arc



from matplotlib.lines import Line2D







from vizlib.animation_export import export_animation











OUTPUT_FORMAT = "webm"   # webm | mp4 | gif



ALPHA = False







OUT_DIR = Path("media-site/animations/HUD/Holographic_Grid")







W, H = 16, 9



DPI = 120



FPS = 24



DURATION = 6



FRAMES = FPS * DURATION







COL = "#35f6ff"



COL_DIM = "#1b7f88"



COL_SOFT = "#9ffcff"



COL_WARN = "#ff4d4d"







LOOPS_DRIFT = 1



LOOPS_PULSE = 4



LOOPS_STARS = 3



LOOPS_SWEEP = 2



LOOPS_GRID = 2











fig, ax = plt.subplots(figsize=(W, H), dpi=DPI)







if ALPHA:



    fig.patch.set_facecolor((0, 0, 0, 0))



    fig.patch.set_alpha(0.0)



    ax.set_facecolor((0, 0, 0, 0))



else:



    fig.patch.set_facecolor("black")



    ax.set_facecolor("black")







ax.set_xlim(-16, 16)



ax.set_ylim(-9, 9)



ax.set_aspect("equal")



ax.axis("off")











rng = np.random.default_rng(123)











# Starfield



stars = []



star_defs = []







for _ in range(140):



    x = rng.uniform(-15.5, 15.5)



    y = rng.uniform(-8.2, 8.2)



    size = rng.uniform(0.7, 2.0)



    phase = rng.uniform(0, 2 * np.pi)



    layer = rng.choice([0.35, 0.6, 1.0], p=[0.45, 0.35, 0.20])







    star, = ax.plot([x], [y], "o", ms=size, color=COL_SOFT, alpha=0.0)



    stars.append(star)



    star_defs.append((x, y, size, phase, layer))











# Perspective grid



grid_lines = []







# Horizontal lines



for i in range(13):



    y = -7.2 + i * 1.2



    line = Line2D(



        [-14.8, 14.8],



        [y, y],



        lw=0.45,



        color=COL_DIM,



        alpha=0.12,



    )



    ax.add_line(line)



    grid_lines.append(("h", line, y))







# Vertical lines



for i in range(19):



    x = -14.4 + i * 1.6



    line = Line2D(



        [x, x],



        [-7.6, 7.6],



        lw=0.45,



        color=COL_DIM,



        alpha=0.12,



    )



    ax.add_line(line)



    grid_lines.append(("v", line, x))











# Range rings / holographic target area



CENTER = (0.0, 0.0)







rings = []



for r in [1.6, 3.2, 4.8, 6.4]:



    ring = Circle(CENTER, r, fill=False, ec=COL_DIM, lw=0.7, alpha=0.18)



    ax.add_patch(ring)



    rings.append(ring)











# Rotating arcs



arcs = []



for radius, theta1, theta2, alpha in [



    (6.9, 12, 85, 0.65),



    (6.9, 192, 265, 0.65),



    (4.1, 110, 155, 0.45),



    (4.1, 290, 335, 0.45),



]:



    arc = Arc(



        CENTER,



        radius * 2,



        radius * 2,



        angle=0,



        theta1=theta1,



        theta2=theta2,



        lw=1.0,



        color=COL,



        alpha=alpha,



    )



    ax.add_patch(arc)



    arcs.append(arc)











# Sweep beam



sweep_line = Line2D([], [], lw=1.0, color=COL, alpha=0.7)



ax.add_line(sweep_line)







sweep_dot = Circle(CENTER, 0.08, fill=True, color=COL_WARN, alpha=0.0)



ax.add_patch(sweep_dot)











# Data nodes



node_defs = [



    (-8.5, 3.2, "N-01"),



    (-4.8, -2.1, "N-02"),



    (3.1, 2.7, "N-03"),



    (7.6, -1.4, "N-04"),



    (10.5, 4.9, "N-05"),



]







nodes = []



node_labels = []







for x, y, label in node_defs:



    dot = Circle((x, y), 0.08, fill=True, color=COL, alpha=0.5)



    halo = Circle((x, y), 0.32, fill=False, ec=COL, lw=0.7, alpha=0.2)







    ax.add_patch(halo)



    ax.add_patch(dot)







    txt = ax.text(



        x + 0.22,



        y + 0.12,



        label,



        color=COL_DIM,



        fontsize=7,



        family="monospace",



        alpha=0.55,



    )







    nodes.append((dot, halo))



    node_labels.append(txt)











# Connecting faint network lines



network_lines = []



connections = [(0, 1), (1, 2), (2, 3), (3, 4)]







for a_idx, b_idx in connections:



    x0, y0, _ = node_defs[a_idx]



    x1, y1, _ = node_defs[b_idx]







    line = Line2D([x0, x1], [y0, y1], lw=0.7, color=COL_DIM, alpha=0.18)



    ax.add_line(line)



    network_lines.append(line)











# Telemetry



txt_title = ax.text(



    -14.7,



    7.55,



    "HOLOGRAPHIC GRID // STARFIELD REGISTRATION",



    color=COL,



    fontsize=12,



    family="monospace",



    alpha=0.95,



)







txt_state = ax.text(



    -14.7,



    7.0,



    "",



    color=COL_DIM,



    fontsize=9,



    family="monospace",



    alpha=0.85,



)







txt_lock = ax.text(



    7.2,



    -7.5,



    "",



    color=COL_SOFT,



    fontsize=9,



    family="monospace",



    alpha=0.85,



)











def wrap_range(value: float, min_v: float, max_v: float) -> float:



    span = max_v - min_v



    return ((value - min_v) % span) + min_v











def update(frame_idx: int) -> None:



    t = frame_idx / FRAMES



    tau = 2 * np.pi







    pulse = 0.5 + 0.5 * np.sin(tau * LOOPS_PULSE * t)



    drift = np.sin(tau * LOOPS_DRIFT * t)



    sweep_angle = 360 * LOOPS_SWEEP * t







    # Starfield parallax drift



    for i, star in enumerate(stars):



        x, y, _, phase, layer = star_defs[i]







        dx = 0.45 * layer * np.sin(tau * LOOPS_DRIFT * t)



        dy = 0.22 * layer * np.cos(tau * LOOPS_DRIFT * t)







        sx = wrap_range(x + dx, -15.5, 15.5)



        sy = wrap_range(y + dy, -8.2, 8.2)







        flicker = 0.5 + 0.5 * np.sin(tau * LOOPS_STARS * t + phase)







        star.set_data([sx], [sy])



        star.set_alpha(0.05 + 0.42 * flicker * layer)







    # Grid breathing/drifting



    for i, (kind, line, base) in enumerate(grid_lines):



        phase = tau * LOOPS_GRID * t + i * 0.12



        alpha = 0.035 + 0.16 * (0.5 + 0.5 * np.sin(phase))







        if kind == "h":



            y = base + 0.16 * drift



            line.set_data([-14.8, 14.8], [y, y])



        else:



            x = base + 0.22 * drift



            line.set_data([x, x], [-7.6, 7.6])







        line.set_alpha(alpha)







    # Range rings



    for i, ring in enumerate(rings):



        phase = tau * LOOPS_GRID * t + i * 0.7



        ring.set_alpha(0.06 + 0.24 * (0.5 + 0.5 * np.sin(phase)))



        ring.set_radius([1.6, 3.2, 4.8, 6.4][i] + 0.08 * pulse)







    # Rotating arcs



    for i, arc in enumerate(arcs):



        arc.angle = 360 * (LOOPS_SWEEP + i * 0.5) * t



        arc.set_alpha(0.22 + 0.5 * pulse)







    # Sweep



    a = np.deg2rad(sweep_angle)



    sx = 6.8 * np.cos(a)



    sy = 6.8 * np.sin(a)







    sweep_line.set_data([0, sx], [0, sy])



    sweep_line.set_alpha(0.18 + 0.5 * pulse)







    sweep_dot.center = (sx, sy)



    sweep_dot.set_alpha(0.25 + 0.55 * pulse)







    # Nodes



    for i, ((dot, halo), label) in enumerate(zip(nodes, node_labels)):



        phase = tau * LOOPS_STARS * t + i * 0.8



        k = 0.5 + 0.5 * np.sin(phase)







        dot.set_alpha(0.25 + 0.6 * k)



        halo.set_alpha(0.08 + 0.38 * k)



        halo.set_radius(0.22 + 0.22 * k)



        label.set_alpha(0.2 + 0.45 * k)







    for i, line in enumerate(network_lines):



        phase = tau * LOOPS_GRID * t + i * 0.7



        line.set_alpha(0.05 + 0.25 * (0.5 + 0.5 * np.sin(phase)))







    # Telemetry



    parallax = 0.42 + 0.08 * np.sin(tau * LOOPS_DRIFT * t)



    lock = 87 + int(9 * pulse)







    txt_state.set_text(



        f"GRID PHASE {sweep_angle:06.2f} DEG  PARALLAX {parallax:04.2f}  NODES {len(nodes):02d}"



    )







    txt_lock.set_text(



        f"REGISTRATION LOCK {lock:02d}%  CELESTIAL FRAME STABLE"



    )







    txt_title.set_alpha(0.68 + 0.28 * pulse)











frames = []







for frame_idx in range(FRAMES):



    update(frame_idx)



    fig.canvas.draw()







    frame_rgba = np.asarray(fig.canvas.renderer.buffer_rgba())

    if ALPHA:

        frames.append(frame_rgba.copy())

    else:

        frames.append(frame_rgba[:, :, :3].copy())







plt.close(fig)











out_file = export_animation(



    frames=frames,



    out_dir=OUT_DIR,



    animation_name="holographic_grid_starfield_v1",



    output_format=OUTPUT_FORMAT,



    fps=FPS,



    alpha=ALPHA,



)







print(f"Saved: {out_file}")



print(f"Frames: {len(frames)}")



print(f"Size: {out_file.stat().st_size / 1024:.1f} KB")



ffmpeg version 7.1.1 Copyright (c) 2000-2025 the FFmpeg developers
  built with clang version 18.1.8
  configuration: --prefix=/Users/mloktionov/anaconda3/envs/astro-ai --cc=arm64-apple-darwin20.0.0-clang --cxx=arm64-apple-darwin20.0.0-clang++ --nm=arm64-apple-darwin20.0.0-nm --ar=arm64-apple-darwin20.0.0-ar --disable-doc --enable-openssl --enable-demuxer=dash --enable-hardcoded-tables --enable-libfreetype --enable-libharfbuzz --enable-libfontconfig --enable-libopenh264 --enable-libdav1d --enable-cross-compile --arch=arm64 --target-os=darwin --cross-prefix=arm64-apple-darwin20.0.0- --host-cc=/Users/runner/miniforge3/conda-bld/ffmpeg_1748704173249/_build_env/bin/x86_64-apple-darwin13.4.0-clang --enable-neon --disable-gnutls --enable-libvpx --enable-libass --enable-pthreads --enable-libopenvino --enable-gpl --enable-libx264 --enable-libx265 --enable-libmp3lame --enable-libaom --enable-libsvtav1 --enable-libxml2 --enable-pic --enable-shared --disable-static --enable-version3 --enable-zlib

Saved: animations/HUD/Holographic_Grid/holographic_grid_starfield_v1.webm
Frames: 144
Size: 2044.8 KB
